# Notebook 1 — Open-source genetic and pathway data collection for medicinal-value prediction

This revised notebook builds the **open-source genetic/pathway evidence layer** for a later ML model predicting or prioritizing medicinal value in wild plants.

It is designed for projects where new laboratory genetic analyses are not feasible. Instead of generating new DNA barcodes, SNPs, genomes or transcriptomes, it collects and harmonizes existing public evidence from open databases.

## What this notebook does

It collects and derives:

- NCBI Taxonomy IDs for target taxa.
- PubChem compound identifiers, with correct handling of broad compound classes.
- KEGG biochemical/pathway context.
- NCBI Nucleotide barcode, ITS, cpDNA and plastid/chloroplast sequence manifests.
- NCBI Assembly metadata using an **automatic Entrez fallback** when the NCBI Datasets CLI is unavailable.
- NCBI Protein and UniProt candidate pathway protein evidence.
- ENA sequencing and transcriptome metadata.
- GBIF occurrence records with **liberal and strict coordinate filters**.
- PubMed literature manifests for metabolite, pathway and accumulation evidence.
- Evidence-tiered candidate gene/pathway tables.
- A taxon-level open genetic feature matrix.
- A taxon–compound target seed table.
- Graph-ready node and edge tables for later GNN/Graph Transformer work.
- A clean ML handoff manifest.

## What this notebook does not do

It does not train ML models.  
It does not prove causal environment–gene–metabolite mechanisms.  
It does not claim that open public genetic data represent exact local populations unless accession metadata support that claim.

## Main scientific interpretation

Public genetic data should be interpreted as:

1. taxon-level genetic/pathway potential;
2. public marker/barcode evidence;
3. genome/transcriptome/proteome availability;
4. candidate pathway evidence with evidence tiers;
5. graph priors linking taxa, compounds, pathways and gene families.

The next notebook should build the environmental + target ML table.

## 0. Recommended environment

Recommended:

```bash
conda create -n open-genetic-medicinal python=3.11 -y
conda activate open-genetic-medicinal

conda install -c conda-forge pandas numpy requests biopython tqdm pyarrow pyyaml scikit-learn -y
conda install -c bioconda entrez-direct seqkit diamond hmmer -y
```

Optional:

```bash
conda install -c conda-forge ncbi-datasets-cli -y
```

This revised notebook does **not require** `ncbi-datasets-cli`; if unavailable, it uses Entrez Assembly automatically.

In [ ]:
# ============================================================
# 0. Imports
# ============================================================

import os
import re
import io
import sys
import json
import time
import gzip
import shutil
import zipfile
import hashlib
import pathlib
import datetime as dt
import subprocess
from typing import Dict, List, Optional, Iterable, Tuple

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

try:
    from Bio import Entrez, SeqIO
except Exception as e:
    Entrez = None
    SeqIO = None
    print("Biopython is required for Entrez and FASTA parsing:", e)

## 1. Configuration

Edit `TARGET_TAXA`, `TARGET_COMPOUNDS`, and `PATHWAY_GENE_SETS` before running.

Important design rules:

- Use genus/species rows as modelling candidates.
- Use family rows as hierarchy/background nodes only.
- Treat compound classes differently from single molecular compounds.
- Do metadata-only runs before enabling large downloads.

In [ ]:
# ============================================================
# 1. Project configuration
# ============================================================

CONFIG = {
    "project_dir": "open_source_genetic_medicinal_value_dataset",

    # Required by NCBI Entrez responsible-use conventions.
    "entrez_email": "your.email@example.org",
    "entrez_tool": "open_source_genetic_medicinal_value_dataset",
    "ncbi_api_key": os.environ.get("NCBI_API_KEY", ""),

    # Safe first-run limits.
    "max_entrez_records_per_query": 1000,
    "entrez_batch_size": 200,
    "entrez_sleep_seconds": 0.34,
    "ena_limit_per_taxon": 5000,
    "uniprot_size_per_query": 100,
    "gbif_occurrence_limit_per_taxon": 1000,
    "gbif_occurrence_country": "",

    # Download switches. Keep False for first run.
    "download_barcode_fastas": False,
    "download_chloroplast_fastas": False,
    "download_ncbi_genome_packages": False,
    "download_uniprot_seed_fastas": True,

    # Optional NCBI Datasets package settings.
    "ncbi_datasets_include": "genome,gff3,gtf,cds,rna,protein,gbff,seq-report",
    "ncbi_assembly_source": "all",
    "ncbi_assembly_level": "",
    "ncbi_exclude_atypical": True,

    # Barcode / marker loci.
    "barcode_markers": [
        "rbcL", "matK", "ITS", "ITS1", "ITS2",
        "trnH-psbA", "psbA-trnH", "trnL-F", "trnL", "trnF",
        "ycf1", "ndhF", "atpB-rbcL"
    ],

    # ENA sequencing strategies.
    "ena_library_strategy_terms": [
        "RNA-Seq", "WGS", "AMPLICON", "Targeted-Capture", "OTHER"
    ],

    # GBIF coordinate filters.
    "gbif_strict_coordinate_uncertainty_m": 1000,
    "gbif_liberal_coordinate_uncertainty_m": 10000,
    "gbif_preferred_basis_of_record": ["PRESERVED_SPECIMEN", "MATERIAL_SAMPLE", "LIVING_SPECIMEN"],
    "gbif_background_basis_of_record": ["HUMAN_OBSERVATION", "OBSERVATION", "MACHINE_OBSERVATION"],

    # Later modelling/grouping.
    "spatial_block_degrees": 5.0,
    "random_seed": 42,
}

TARGET_TAXA = [
    # Project-relevant genera
    {"taxon_id": "TAX_SEDUM", "scientific_name": "Sedum", "rank": "genus", "family": "Crassulaceae", "priority": "project_core", "ml_role": "candidate_taxon"},
    {"taxon_id": "TAX_RHODIOLA", "scientific_name": "Rhodiola", "rank": "genus", "family": "Crassulaceae", "priority": "project_core", "ml_role": "candidate_taxon"},
    {"taxon_id": "TAX_HYPERICUM", "scientific_name": "Hypericum", "rank": "genus", "family": "Hypericaceae", "priority": "project_core", "ml_role": "candidate_taxon"},

    # Families from project concept. Use as hierarchy/background only.
    {"taxon_id": "TAX_ASTERACEAE", "scientific_name": "Asteraceae", "rank": "family", "family": "Asteraceae", "priority": "project_family", "ml_role": "hierarchy_context"},
    {"taxon_id": "TAX_CRASSULACEAE", "scientific_name": "Crassulaceae", "rank": "family", "family": "Crassulaceae", "priority": "project_family", "ml_role": "hierarchy_context"},
    {"taxon_id": "TAX_NITRARIACEAE", "scientific_name": "Nitrariaceae", "rank": "family", "family": "Nitrariaceae", "priority": "project_family", "ml_role": "hierarchy_context"},
    {"taxon_id": "TAX_OROBANCHACEAE", "scientific_name": "Orobanchaceae", "rank": "family", "family": "Orobanchaceae", "priority": "project_family", "ml_role": "hierarchy_context"},
    {"taxon_id": "TAX_POLYGONACEAE", "scientific_name": "Polygonaceae", "rank": "family", "family": "Polygonaceae", "priority": "project_family", "ml_role": "hierarchy_context"},

    # Positive-control or universal medicinal-plant examples.
    {"taxon_id": "TAX_ARTEMISIA_ANNUA", "scientific_name": "Artemisia annua", "rank": "species", "family": "Asteraceae", "priority": "positive_control", "ml_role": "candidate_taxon"},
    {"taxon_id": "TAX_GLYCYRRHIZA_GLABRA", "scientific_name": "Glycyrrhiza glabra", "rank": "species", "family": "Fabaceae", "priority": "positive_control", "ml_role": "candidate_taxon"},
    {"taxon_id": "TAX_TAXUS", "scientific_name": "Taxus", "rank": "genus", "family": "Taxaceae", "priority": "positive_control", "ml_role": "candidate_taxon"},
]

TARGET_COMPOUNDS = [
    {"compound_id": "CMP_SALIDROSIDE", "compound_name": "salidroside", "compound_class": "phenolic glycoside", "record_type": "molecule", "relevant_taxa": "Rhodiola; Crassulaceae"},
    {"compound_id": "CMP_TYROSOL", "compound_name": "tyrosol", "compound_class": "phenolic alcohol", "record_type": "molecule", "relevant_taxa": "Rhodiola; Olea; broad plants"},
    {"compound_id": "CMP_HYPERICIN", "compound_name": "hypericin", "compound_class": "naphthodianthrone", "record_type": "molecule", "relevant_taxa": "Hypericum"},
    {"compound_id": "CMP_ARTEMISININ", "compound_name": "artemisinin", "compound_class": "sesquiterpene lactone", "record_type": "molecule", "relevant_taxa": "Artemisia"},
    {"compound_id": "CMP_GLYCYRRHIZIN", "compound_name": "glycyrrhizin", "compound_class": "triterpenoid saponin", "record_type": "molecule", "relevant_taxa": "Glycyrrhiza"},
    {"compound_id": "CMP_PACLITAXEL", "compound_name": "paclitaxel", "compound_class": "diterpenoid taxane", "record_type": "molecule", "relevant_taxa": "Taxus"},
    {"compound_id": "CMP_TOTAL_FLAVONOIDS", "compound_name": "flavonoids", "compound_class": "flavonoids", "record_type": "compound_class", "relevant_taxa": "broad plants"},
    {"compound_id": "CMP_TOTAL_PHENOLICS", "compound_name": "phenolic compounds", "compound_class": "phenolic compounds", "record_type": "compound_class", "relevant_taxa": "broad plants"},
    {"compound_id": "CMP_ALKALOIDS", "compound_name": "alkaloids", "compound_class": "alkaloids", "record_type": "compound_class", "relevant_taxa": "broad plants"},
    {"compound_id": "CMP_TERPENOIDS", "compound_name": "terpenoids", "compound_class": "terpenoids", "record_type": "compound_class", "relevant_taxa": "broad plants"},
]

PATHWAY_GENE_SETS = [
    {
        "pathway_id": "PWY_SALIDROSIDE",
        "compound_ids": "CMP_SALIDROSIDE;CMP_TYROSOL",
        "pathway_name": "salidroside / tyrosol glycoside biosynthesis",
        "gene_families": "4HPAAS;4HPAR;T8GT;UGT",
        "query_terms": [
            "4-hydroxyphenylacetaldehyde synthase",
            "aromatic aldehyde synthase",
            "4-hydroxyphenylacetaldehyde reductase",
            "tyrosol glucosyltransferase",
            "UDP-glycosyltransferase",
            "UGT73B6"
        ],
        "notes": "Specific evidence strongest in Rhodiola. Generic UGT alone is weak."
    },
    {
        "pathway_id": "PWY_PHENYLPROPANOID_FLAVONOID",
        "compound_ids": "CMP_TOTAL_FLAVONOIDS;CMP_TOTAL_PHENOLICS",
        "pathway_name": "phenylpropanoid and flavonoid biosynthesis",
        "gene_families": "PAL;C4H;4CL;CHS;CHI;F3H;FLS;DFR;ANS;UGT;OMT",
        "query_terms": [
            "phenylalanine ammonia-lyase",
            "cinnamate 4-hydroxylase",
            "4-coumarate-CoA ligase",
            "chalcone synthase",
            "chalcone isomerase",
            "flavanone 3-hydroxylase",
            "flavonol synthase",
            "dihydroflavonol reductase",
            "anthocyanidin synthase",
            "UDP-glycosyltransferase",
            "O-methyltransferase"
        ],
        "notes": "Broad pathway relevant to many medicinal plant classes."
    },
    {
        "pathway_id": "PWY_TERPENOID",
        "compound_ids": "CMP_ARTEMISININ;CMP_PACLITAXEL;CMP_TERPENOIDS;CMP_GLYCYRRHIZIN",
        "pathway_name": "terpenoid and triterpenoid biosynthesis",
        "gene_families": "DXS;DXR;HMGR;TPS;CYP450;DBAT;BAPT;UGT;OSC",
        "query_terms": [
            "terpene synthase",
            "cytochrome P450",
            "1-deoxy-D-xylulose-5-phosphate synthase",
            "1-deoxy-D-xylulose 5-phosphate reductoisomerase",
            "3-hydroxy-3-methylglutaryl-CoA reductase",
            "amorpha-4,11-diene synthase",
            "taxadiene synthase",
            "10-deacetylbaccatin III-10-O-acetyltransferase",
            "UDP-glycosyltransferase",
            "beta-amyrin synthase",
            "oxidosqualene cyclase"
        ],
        "notes": "Generic TPS/CYP450/UGT queries are broad; use evidence tiers."
    },
    {
        "pathway_id": "PWY_ALKALOID",
        "compound_ids": "CMP_ALKALOIDS",
        "pathway_name": "alkaloid biosynthesis",
        "gene_families": "NCS;STR;BBE;OMT;CYP450;AAT;TDC",
        "query_terms": [
            "norcoclaurine synthase",
            "strictosidine synthase",
            "berberine bridge enzyme",
            "O-methyltransferase",
            "cytochrome P450",
            "aromatic amino acid decarboxylase",
            "tryptophan decarboxylase"
        ],
        "notes": "Broad alkaloid framework; exact gene set should be specialized by compound."
    },
    {
        "pathway_id": "PWY_HYPERICIN",
        "compound_ids": "CMP_HYPERICIN",
        "pathway_name": "hypericin / naphthodianthrone-related biosynthesis",
        "gene_families": "PKS;Hyp-1;CYP450;POX;UGT",
        "query_terms": [
            "type III polyketide synthase",
            "Hyp-1",
            "hypericin biosynthesis",
            "phenolic oxidative coupling",
            "cytochrome P450",
            "peroxidase",
            "UDP-glycosyltransferase"
        ],
        "notes": "Hypericum-specific evidence requires manual curation."
    },
]

PROJECT = pathlib.Path(CONFIG["project_dir"]).resolve()
DIRS = {
    "metadata": PROJECT / "metadata",
    "taxonomy": PROJECT / "taxonomy",
    "compounds": PROJECT / "compounds",
    "pathways": PROJECT / "pathways",
    "ncbi": PROJECT / "ncbi",
    "uniprot": PROJECT / "uniprot",
    "ena": PROJECT / "ena",
    "gbif": PROJECT / "gbif",
    "literature": PROJECT / "literature",
    "derived": PROJECT / "derived",
    "logs": PROJECT / "logs",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
CONFIG["run_id"] = RUN_ID
CONFIG["retrieved_utc"] = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")

pd.DataFrame(TARGET_TAXA).to_csv(DIRS["metadata"] / "target_taxa_config.tsv", sep="\t", index=False)
pd.DataFrame(TARGET_COMPOUNDS).to_csv(DIRS["metadata"] / "target_compounds_config.tsv", sep="\t", index=False)
pd.DataFrame(PATHWAY_GENE_SETS).to_csv(DIRS["metadata"] / "pathway_gene_sets_config.tsv", sep="\t", index=False)
with open(PROJECT / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

PROJECT

## 2. Source registry

In [ ]:
# ============================================================
# 2. Source registry
# ============================================================

SOURCE_REGISTRY = pd.DataFrame([
    {"source_id": "NCBI_TAXONOMY", "source_name": "NCBI Taxonomy via Entrez", "role": "resolve taxon names to NCBI taxids", "output": "taxonomy/taxon_resolution.tsv", "access": "Entrez E-utilities"},
    {"source_id": "NCBI_NUCCORE", "source_name": "NCBI Nucleotide / GenBank", "role": "barcodes, ITS/cpDNA markers, plastomes, chloroplast genomes", "output": "ncbi/barcode_sequence_manifest.tsv; ncbi/chloroplast_plastid_manifest.tsv", "access": "Entrez E-utilities"},
    {"source_id": "NCBI_ASSEMBLY", "source_name": "NCBI Assembly via Entrez", "role": "genome assembly metadata fallback when Datasets CLI unavailable", "output": "ncbi/ncbi_assemblies_manifest.tsv", "access": "Entrez E-utilities"},
    {"source_id": "NCBI_DATASETS", "source_name": "NCBI Datasets", "role": "optional genome package download commands", "output": "ncbi/genome_package_download_commands.tsv", "access": "datasets CLI if available"},
    {"source_id": "ENA", "source_name": "European Nucleotide Archive", "role": "RNA-seq, WGS, amplicon and other sequencing metadata", "output": "ena/ena_run_manifest.tsv", "access": "ENA Portal API"},
    {"source_id": "UNIPROT", "source_name": "UniProtKB", "role": "reference enzymes, protein annotations, pathway seed proteins", "output": "uniprot/uniprot_pathway_seed_proteins.tsv", "access": "UniProt REST API"},
    {"source_id": "NCBI_PROTEIN", "source_name": "NCBI Protein via Entrez", "role": "candidate enzyme/protein manifest by pathway query", "output": "ncbi/ncbi_candidate_protein_manifest.tsv", "access": "Entrez E-utilities"},
    {"source_id": "PUBCHEM", "source_name": "PubChem", "role": "compound identifiers, synonyms, SMILES, InChIKey for molecule records", "output": "compounds/compound_dictionary.tsv", "access": "PUG REST"},
    {"source_id": "KEGG", "source_name": "KEGG REST", "role": "compound, reaction, enzyme, KO and pathway context", "output": "pathways/kegg_context.tsv", "access": "KEGG REST"},
    {"source_id": "GBIF", "source_name": "GBIF", "role": "occurrence and geography context for taxa", "output": "gbif/gbif_occurrence_manifest.tsv; filtered GBIF tables", "access": "GBIF occurrence API"},
    {"source_id": "LOTUS", "source_name": "LOTUS natural products database", "role": "taxon-compound occurrence evidence placeholder", "output": "compounds/lotus_source_manifest.tsv", "access": "download/API where available"},
    {"source_id": "PUBMED", "source_name": "PubMed via Entrez", "role": "literature manifest for metabolite accumulation, gene/pathway and environmental association curation", "output": "literature/pubmed_metabolite_genetic_literature_manifest.tsv", "access": "Entrez E-utilities"},
])
SOURCE_REGISTRY.to_csv(DIRS["metadata"] / "source_registry.tsv", sep="\t", index=False)
SOURCE_REGISTRY

## 3. Utility functions

In [ ]:
# ============================================================
# 3. Utility functions
# ============================================================

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": f"{CONFIG['entrez_tool']}/1.0 ({CONFIG['entrez_email']})"
})

def utc_now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")

def safe_name(x: str, max_len: int = 180) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(x)).strip("_")[:max_len]

def sha256_file(path: pathlib.Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def request_get(url: str, params: Optional[dict] = None, retries: int = 5, sleep: float = 1.0, timeout: int = 120, stream: bool = False) -> requests.Response:
    last = None
    for i in range(retries):
        try:
            r = SESSION.get(url, params=params, timeout=timeout, stream=stream)
            if r.status_code in (429, 500, 502, 503, 504):
                last = r
                time.sleep(sleep * (2 ** i))
                continue
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            time.sleep(sleep * (2 ** i))
    raise RuntimeError(f"GET failed after {retries} attempts: {url} params={params} last={last}")

def write_table(df: pd.DataFrame, path_stem: pathlib.Path) -> Dict[str, str]:
    path_stem.parent.mkdir(parents=True, exist_ok=True)
    tsv = path_stem.with_suffix(".tsv")
    df.to_csv(tsv, sep="\t", index=False)
    out = {"tsv": str(tsv)}
    try:
        pq = path_stem.with_suffix(".parquet")
        df.to_parquet(pq, index=False)
        out["parquet"] = str(pq)
    except Exception as e:
        print(f"Parquet skipped for {path_stem.name}: {e}")
    return out

def load_tsv_if_valid(path: pathlib.Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, sep="\t", dtype=str, low_memory=False)
    except Exception:
        return pd.DataFrame()

def add_provenance(df: pd.DataFrame, source: str, query: str) -> pd.DataFrame:
    df = df.copy()
    df["source"] = source
    df["query"] = query
    df["retrieved_utc"] = utc_now()
    df["run_id"] = RUN_ID
    return df

def executable_exists(name: str) -> bool:
    return shutil.which(name) is not None

def run_cmd(cmd: List[str], log_name: Optional[str] = None, check: bool = True) -> subprocess.CompletedProcess:
    print(" ".join(map(str, cmd)))
    res = subprocess.run(cmd, text=True, capture_output=True)
    if log_name:
        log = DIRS["logs"] / f"{safe_name(log_name)}_{RUN_ID}.log"
        log.write_text(
            "$ " + " ".join(map(str, cmd)) + "\n\nSTDOUT\n------\n" + (res.stdout or "") +
            "\n\nSTDERR\n------\n" + (res.stderr or ""),
            encoding="utf-8"
        )
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(map(str, cmd))}\n{res.stderr}")
    return res

## 4. Resolve taxon names with NCBI Taxonomy

In [ ]:
# ============================================================
# 4. Resolve taxon names with NCBI Taxonomy
# ============================================================

NCBI_EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

if Entrez is not None:
    Entrez.email = CONFIG["entrez_email"]
    Entrez.tool = CONFIG["entrez_tool"]
    if CONFIG.get("ncbi_api_key"):
        Entrez.api_key = CONFIG["ncbi_api_key"]

def entrez_params(extra: Dict) -> Dict:
    p = dict(extra)
    p["tool"] = CONFIG["entrez_tool"]
    p["email"] = CONFIG["entrez_email"]
    if CONFIG.get("ncbi_api_key"):
        p["api_key"] = CONFIG["ncbi_api_key"]
    return p

def ncbi_esearch(db: str, term: str, retmax: int = 0, retstart: int = 0) -> Dict:
    p = entrez_params({
        "db": db, "term": term, "retmode": "json",
        "retmax": retmax, "retstart": retstart
    })
    r = request_get(f"{NCBI_EUTILS}/esearch.fcgi", params=p)
    time.sleep(CONFIG["entrez_sleep_seconds"])
    return r.json()["esearchresult"]

def ncbi_esummary(db: str, ids: List[str]) -> Dict:
    if not ids:
        return {}
    p = entrez_params({"db": db, "id": ",".join(map(str, ids)), "retmode": "json"})
    r = request_get(f"{NCBI_EUTILS}/esummary.fcgi", params=p)
    time.sleep(CONFIG["entrez_sleep_seconds"])
    return r.json()["result"]

def ncbi_efetch_text(db: str, ids: List[str], rettype: str, retmode: str = "text") -> str:
    p = entrez_params({
        "db": db, "id": ",".join(map(str, ids)),
        "rettype": rettype, "retmode": retmode
    })
    r = request_get(f"{NCBI_EUTILS}/efetch.fcgi", params=p, timeout=300)
    time.sleep(CONFIG["entrez_sleep_seconds"])
    return r.text

def resolve_ncbi_taxon(name: str) -> Dict:
    term = f'"{name}"[All Names]'
    sr = ncbi_esearch("taxonomy", term, retmax=5)
    ids = sr.get("idlist", [])
    if not ids:
        sr = ncbi_esearch("taxonomy", name, retmax=5)
        ids = sr.get("idlist", [])
    if not ids:
        return {"scientific_name": name, "ncbi_taxid": None, "ncbi_status": "not_found"}
    sm = ncbi_esummary("taxonomy", [ids[0]])
    rec = sm.get(ids[0], {})
    return {
        "scientific_name": name,
        "ncbi_taxid": int(ids[0]),
        "ncbi_scientific_name": rec.get("scientificname"),
        "ncbi_rank": rec.get("rank"),
        "ncbi_division": rec.get("division"),
        "ncbi_lineage": rec.get("lineage"),
        "ncbi_status": "resolved",
    }

taxon_rows = []
for t in TARGET_TAXA:
    rec = dict(t)
    rec.update(resolve_ncbi_taxon(t["scientific_name"]))
    taxon_rows.append(rec)

taxa_df = pd.DataFrame(taxon_rows)
taxa_df = add_provenance(taxa_df, "NCBI Taxonomy via Entrez E-utilities", "target taxa")
write_table(taxa_df, DIRS["taxonomy"] / "taxon_resolution")
taxa_df

## 5. Compound dictionary from PubChem with compound-class handling

PubChem queries are performed only for `record_type == "molecule"`. Broad classes such as `flavonoids`, `phenolic compounds`, `alkaloids`, and `terpenoids` are retained as ontology classes and are not treated as PubChem-resolvable compounds.

In [ ]:
# ============================================================
# 5. Compound dictionary from PubChem with compound-class handling
# ============================================================

PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

def pubchem_name_to_properties(name: str) -> Dict:
    props = "MolecularFormula,MolecularWeight,CanonicalSMILES,IsomericSMILES,InChIKey,IUPACName"
    url = f"{PUBCHEM_BASE}/compound/name/{requests.utils.quote(name)}/property/{props}/JSON"
    try:
        r = request_get(url, timeout=60)
        data = r.json()
        prop = data.get("PropertyTable", {}).get("Properties", [{}])[0]
    except Exception as e:
        prop = {"error": str(e)}
    return prop

def pubchem_synonyms(name: str) -> str:
    url = f"{PUBCHEM_BASE}/compound/name/{requests.utils.quote(name)}/synonyms/JSON"
    try:
        r = request_get(url, timeout=60)
        syns = r.json().get("InformationList", {}).get("Information", [{}])[0].get("Synonym", [])
        return "; ".join(syns[:50])
    except Exception:
        return ""

compound_rows = []
for c in TARGET_COMPOUNDS:
    row = dict(c)
    record_type = c.get("record_type", "molecule")

    if record_type == "molecule":
        props = pubchem_name_to_properties(c["compound_name"])
        row.update({
            "pubchem_query_attempted": True,
            "pubchem_cid": props.get("CID", ""),
            "molecular_formula": props.get("MolecularFormula", ""),
            "molecular_weight": props.get("MolecularWeight", ""),
            "canonical_smiles": props.get("CanonicalSMILES", ""),
            "isomeric_smiles": props.get("IsomericSMILES", ""),
            "inchikey": props.get("InChIKey", ""),
            "iupac_name": props.get("IUPACName", ""),
            "pubchem_synonyms": pubchem_synonyms(c["compound_name"]),
            "pubchem_error": props.get("error", ""),
            "compound_dictionary_status": "molecule_pubchem_resolved" if props.get("CID", "") else "molecule_pubchem_unresolved",
        })
        time.sleep(0.2)
    else:
        row.update({
            "pubchem_query_attempted": False,
            "pubchem_cid": "",
            "molecular_formula": "",
            "molecular_weight": "",
            "canonical_smiles": "",
            "isomeric_smiles": "",
            "inchikey": "",
            "iupac_name": "",
            "pubchem_synonyms": "",
            "pubchem_error": "Not queried: broad compound class, not a single molecule.",
            "compound_dictionary_status": "compound_class_no_pubchem_resolution_expected",
        })

    compound_rows.append(row)

compound_df = pd.DataFrame(compound_rows)
compound_df = add_provenance(compound_df, "PubChem PUG REST plus local compound-class ontology", "target compounds")
write_table(compound_df, DIRS["compounds"] / "compound_dictionary")
compound_df

## 6. KEGG biochemical context

In [ ]:
# ============================================================
# 6. KEGG biochemical context
# ============================================================

KEGG_BASE = "https://rest.kegg.jp"

def kegg_find(database: str, term: str) -> List[Dict]:
    path = f"find/{database}/{term.replace(' ', '+')}"
    try:
        txt = request_get(f"{KEGG_BASE}/{path}", timeout=90).text
        rows = []
        for line in txt.strip().splitlines():
            if "\t" in line:
                entry, desc = line.split("\t", 1)
            else:
                entry, desc = line, ""
            if entry:
                rows.append({"database": database, "term": term, "entry": entry, "description": desc, "kegg_path": path})
        return rows
    except Exception as e:
        return [{"database": database, "term": term, "entry": "", "description": f"ERROR: {e}", "kegg_path": path}]

kegg_rows = []
for c in TARGET_COMPOUNDS:
    # Molecular compounds and classes both useful as search terms in KEGG.
    for db in ["compound", "reaction", "enzyme", "ko", "pathway"]:
        kegg_rows.extend(kegg_find(db, c["compound_name"]))
        time.sleep(0.2)

for pwy in PATHWAY_GENE_SETS:
    for term in pwy["query_terms"]:
        for db in ["enzyme", "ko", "pathway"]:
            for row in kegg_find(db, term):
                row["pathway_id"] = pwy["pathway_id"]
                kegg_rows.append(row)
            time.sleep(0.2)

kegg_df = pd.DataFrame(kegg_rows)
if not kegg_df.empty:
    kegg_df = add_provenance(kegg_df, "KEGG REST", "target compounds and pathway gene terms")
write_table(kegg_df, DIRS["pathways"] / "kegg_context")
kegg_df.head(30)

## 7. NCBI Nucleotide: public barcodes, cpDNA and ITS marker sequences

In [ ]:
# ============================================================
# 7. NCBI Nucleotide: public barcode and marker sequence manifest
# ============================================================

def iter_entrez_summaries(db: str, term: str, max_records: int, batch_size: int) -> pd.DataFrame:
    sr0 = ncbi_esearch(db, term, retmax=0)
    count = int(sr0.get("count", 0))
    n = min(count, max_records)
    rows = []
    for start in tqdm(range(0, n, batch_size), desc=f"{db}: {term[:70]}"):
        sr = ncbi_esearch(db, term, retmax=min(batch_size, n - start), retstart=start)
        ids = sr.get("idlist", [])
        if not ids:
            continue
        sm = ncbi_esummary(db, ids)
        for uid in sm.get("uids", []):
            rec = sm.get(uid, {})
            rows.append({
                "uid": uid,
                "accession": rec.get("caption"),
                "accession_version": rec.get("accessionversion"),
                "title": rec.get("title"),
                "organism": rec.get("organism"),
                "taxid": rec.get("taxid"),
                "length_bp": rec.get("slen"),
                "moltype": rec.get("moltype"),
                "biomol": rec.get("biomol"),
                "created": rec.get("createdate"),
                "updated": rec.get("updatedate"),
                "query_total_count": count,
                "records_retrieved_for_query": n,
                "is_query_capped": int(count > max_records),
            })
    return pd.DataFrame(rows)

def barcode_query(taxid: int, marker: str) -> str:
    marker_terms = f'("{marker}"[Title] OR "{marker}"[Gene Name] OR "{marker}"[All Fields])'
    return (
        f"txid{taxid}[Organism:exp] AND {marker_terms} "
        f"NOT mitochondrion[Title] NOT mitochondrial[Title]"
    )

barcode_tables = []
for _, tax in taxa_df.dropna(subset=["ncbi_taxid"]).iterrows():
    for marker in CONFIG["barcode_markers"]:
        q = barcode_query(int(tax["ncbi_taxid"]), marker)
        df = iter_entrez_summaries(
            "nuccore", q,
            CONFIG["max_entrez_records_per_query"],
            CONFIG["entrez_batch_size"]
        )
        if not df.empty:
            df["taxon_id"] = tax["taxon_id"]
            df["input_taxon"] = tax["scientific_name"]
            df["resolved_taxid"] = int(tax["ncbi_taxid"])
            df["resolved_name"] = tax["ncbi_scientific_name"]
            df["marker_query"] = marker
            df = add_provenance(df, "NCBI Nucleotide via Entrez", q)
            barcode_tables.append(df)

barcode_df = pd.concat(barcode_tables, ignore_index=True) if barcode_tables else pd.DataFrame()
if not barcode_df.empty:
    barcode_df = barcode_df.drop_duplicates(subset=["accession_version", "marker_query"])
write_table(barcode_df, DIRS["ncbi"] / "barcode_sequence_manifest")
barcode_df.head()

In [ ]:
# Optional barcode FASTA download.

def efetch_fasta(accession: str, out_path: pathlib.Path, db: str = "nuccore") -> pathlib.Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    txt = ncbi_efetch_text(db, [accession], rettype="fasta", retmode="text")
    out_path.write_text(txt, encoding="utf-8")
    return out_path

if CONFIG["download_barcode_fastas"] and not barcode_df.empty:
    rows = []
    for acc in tqdm(barcode_df["accession_version"].dropna().unique(), desc="Downloading barcode FASTA"):
        out = DIRS["ncbi"] / "barcode_fastas" / f"{safe_name(acc)}.fasta"
        efetch_fasta(acc, out, db="nuccore")
        rows.append({"accession_version": acc, "file": str(out), "bytes": out.stat().st_size, "sha256": sha256_file(out), "retrieved_utc": utc_now(), "run_id": RUN_ID})
    barcode_downloads_df = pd.DataFrame(rows)
    write_table(barcode_downloads_df, DIRS["ncbi"] / "barcode_fasta_downloads")
else:
    print("Barcode FASTA download disabled.")

## 8. NCBI Nucleotide: chloroplast/plastid genome manifests

In [ ]:
# ============================================================
# 8. Chloroplast/plastid manifest
# ============================================================

def chloroplast_query(taxid: int) -> str:
    return (
        f"txid{taxid}[Organism:exp] AND "
        f"(chloroplast[Title] OR plastid[Title]) AND "
        f"(complete genome[Title] OR complete sequence[Title] OR genome[Title]) "
        f"NOT mitochondrion[Title] NOT mitochondrial[Title]"
    )

cp_tables = []
for _, tax in taxa_df.dropna(subset=["ncbi_taxid"]).iterrows():
    q = chloroplast_query(int(tax["ncbi_taxid"]))
    df = iter_entrez_summaries(
        "nuccore", q,
        CONFIG["max_entrez_records_per_query"],
        CONFIG["entrez_batch_size"]
    )
    if not df.empty:
        df["taxon_id"] = tax["taxon_id"]
        df["input_taxon"] = tax["scientific_name"]
        df["resolved_taxid"] = int(tax["ncbi_taxid"])
        df["resolved_name"] = tax["ncbi_scientific_name"]
        df = add_provenance(df, "NCBI Nucleotide via Entrez", q)
        cp_tables.append(df)

chloroplast_df = pd.concat(cp_tables, ignore_index=True) if cp_tables else pd.DataFrame()
if not chloroplast_df.empty:
    chloroplast_df = chloroplast_df.drop_duplicates(subset=["accession_version"])
write_table(chloroplast_df, DIRS["ncbi"] / "chloroplast_plastid_manifest")
chloroplast_df.head()

## 9. Genome assembly metadata

This revised section fixes the previous error where the NCBI Datasets CLI was unavailable. The notebook now:

1. tries NCBI Datasets CLI if available;
2. always creates download commands if CLI is available;
3. uses Entrez Assembly as a robust fallback;
4. interprets annotation metadata as **not extracted** when using Entrez Assembly, not as absent.

In [ ]:
# ============================================================
# 9A. Optional NCBI Datasets CLI summary
# ============================================================

def datasets_summary_genome_taxon(taxon_name: str) -> pd.DataFrame:
    if not executable_exists("datasets"):
        print("NCBI Datasets CLI not found. Entrez Assembly fallback will be used.")
        return pd.DataFrame()
    cmd = ["datasets", "summary", "genome", "taxon", taxon_name]
    if CONFIG.get("ncbi_api_key"):
        cmd.extend(["--api-key", CONFIG["ncbi_api_key"]])
    res = run_cmd(cmd, log_name=f"datasets_summary_{safe_name(taxon_name)}", check=False)
    if res.returncode != 0:
        print(f"datasets summary failed for {taxon_name}: {res.stderr[:300]}")
        return pd.DataFrame()
    try:
        js = json.loads(res.stdout)
    except Exception as e:
        print(f"Could not parse datasets output for {taxon_name}: {e}")
        return pd.DataFrame()
    rows = []
    for item in js.get("reports", []):
        rep = item.get("report", item)
        org = rep.get("organism", {}) or {}
        asm = rep.get("assembly_info", {}) or {}
        ann = rep.get("annotation_info", {}) or {}
        rows.append({
            "input_taxon": taxon_name,
            "assembly_accession": rep.get("accession"),
            "assembly_name": asm.get("assembly_name"),
            "assembly_level": asm.get("assembly_level"),
            "assembly_status": asm.get("assembly_status"),
            "seq_length": rep.get("seq_length"),
            "scaffold_count": rep.get("scaffold_count"),
            "contig_count": rep.get("contig_count"),
            "taxid": org.get("tax_id"),
            "organism_name": org.get("organism_name"),
            "infraspecific_name": org.get("infraspecific_name"),
            "isolate": org.get("isolate"),
            "annotation_name": ann.get("name"),
            "annotation_release_date": ann.get("release_date"),
            "annotation_metadata_status": "extracted_from_ncbi_datasets",
            "bioproject_accession": rep.get("bioproject_accession"),
            "biosample_accession": rep.get("biosample_accession"),
            "submitter": rep.get("submitter"),
            "assembly_metadata_source": "NCBI Datasets CLI",
            "raw_json": json.dumps(rep, sort_keys=True),
        })
    return pd.DataFrame(rows)

assembly_tables = []
if executable_exists("datasets"):
    for _, tax in taxa_df.iterrows():
        df = datasets_summary_genome_taxon(tax["scientific_name"])
        if not df.empty:
            df["taxon_id"] = tax["taxon_id"]
            df = add_provenance(df, "NCBI Datasets CLI", f"datasets summary genome taxon {tax['scientific_name']}")
            assembly_tables.append(df)

datasets_assemblies_df = pd.concat(assembly_tables, ignore_index=True) if assembly_tables else pd.DataFrame()
if not datasets_assemblies_df.empty:
    datasets_assemblies_df = datasets_assemblies_df.drop_duplicates(subset=["assembly_accession", "taxon_id"])
write_table(datasets_assemblies_df, DIRS["ncbi"] / "ncbi_assemblies_manifest_datasets_cli")
print("NCBI Datasets CLI assembly rows:", len(datasets_assemblies_df))
datasets_assemblies_df.head()

In [ ]:
# ============================================================
# 9B. Entrez Assembly fallback / primary repair
# ============================================================

def fetch_assembly_records_for_taxon(taxon_id, taxon_name, ncbi_taxid):
    rows = []
    queries = [
        f"txid{int(ncbi_taxid)}[Organism:exp]",
        f'"{taxon_name}"[Organism]',
        f'"{taxon_name}"[All Fields]',
    ]

    for query in queries:
        try:
            sr = ncbi_esearch(
                db="assembly",
                term=query,
                retmax=CONFIG.get("max_entrez_records_per_query", 1000),
                retstart=0
            )
            ids = sr.get("idlist", [])
            total_count = int(sr.get("count", 0))

            print(f"{taxon_name}: {query} -> count={total_count}, fetched={len(ids)}")

            if not ids:
                continue

            batch_size = CONFIG.get("entrez_batch_size", 200)
            for start in range(0, len(ids), batch_size):
                batch = ids[start:start + batch_size]
                sm = ncbi_esummary("assembly", batch)

                for uid in sm.get("uids", []):
                    rec = sm.get(uid, {})
                    rows.append({
                        "taxon_id": taxon_id,
                        "scientific_name_query": taxon_name,
                        "input_taxon": taxon_name,
                        "taxid": int(ncbi_taxid),
                        "assembly_uid": uid,
                        "assembly_accession": rec.get("assemblyaccession", ""),
                        "assembly_name": rec.get("assemblyname", ""),
                        "assembly_level": rec.get("assemblystatus", ""),
                        "assembly_status": rec.get("assemblystatus", ""),
                        "organism_name": rec.get("organism", ""),
                        "species_taxid": rec.get("species_taxid", ""),
                        "assembly_type": rec.get("assemblytype", ""),
                        "assembly_release_date": rec.get("releasedate", ""),
                        "assembly_update_date": rec.get("lastupdatedate", ""),
                        "bioproject_accession": rec.get("bioproject", ""),
                        "biosample_accession": rec.get("biosampleaccn", ""),
                        "submitter": rec.get("submitterorganization", ""),
                        "refseq_category": rec.get("refseq_category", ""),
                        "ftp_path_genbank": rec.get("ftppath_genbank", ""),
                        "ftp_path_refseq": rec.get("ftppath_refseq", ""),
                        "query_total_count": total_count,
                        "records_retrieved_for_query": len(ids),
                        "is_query_capped": int(total_count > CONFIG.get("max_entrez_records_per_query", 1000)),
                        "annotation_name": "",
                        "annotation_release_date": "",
                        "annotation_metadata_status": "not_extracted_from_entrez_assembly",
                        "assembly_metadata_source": "NCBI Entrez Assembly",
                        "assembly_repair_error": "",
                        "query": query,
                        "retrieved_utc": utc_now(),
                        "run_id": RUN_ID,
                    })

            if rows:
                break

        except Exception as e:
            print(f"ERROR for {taxon_name} / {query}: {e}")

    if not rows:
        rows.append({
            "taxon_id": taxon_id,
            "scientific_name_query": taxon_name,
            "input_taxon": taxon_name,
            "taxid": int(ncbi_taxid),
            "assembly_uid": "",
            "assembly_accession": "",
            "assembly_name": "",
            "assembly_level": "",
            "assembly_status": "",
            "organism_name": "",
            "species_taxid": "",
            "assembly_type": "",
            "assembly_release_date": "",
            "assembly_update_date": "",
            "bioproject_accession": "",
            "biosample_accession": "",
            "submitter": "",
            "refseq_category": "",
            "ftp_path_genbank": "",
            "ftp_path_refseq": "",
            "query_total_count": 0,
            "records_retrieved_for_query": 0,
            "is_query_capped": 0,
            "annotation_name": "",
            "annotation_release_date": "",
            "annotation_metadata_status": "not_extracted_from_entrez_assembly",
            "assembly_metadata_source": "NCBI Entrez Assembly",
            "assembly_repair_error": "No assembly records found",
            "query": " | ".join(queries),
            "retrieved_utc": utc_now(),
            "run_id": RUN_ID,
        })

    return rows

all_rows = []
taxa_tmp = taxa_df.copy()
taxa_tmp["ncbi_taxid"] = pd.to_numeric(taxa_tmp["ncbi_taxid"], errors="coerce")
taxa_tmp = taxa_tmp.dropna(subset=["ncbi_taxid"])

print("Taxa with valid NCBI taxid:", len(taxa_tmp))

for _, tax in tqdm(taxa_tmp.iterrows(), total=len(taxa_tmp), desc="Fetching NCBI Assembly records"):
    all_rows.extend(fetch_assembly_records_for_taxon(tax["taxon_id"], tax["scientific_name"], tax["ncbi_taxid"]))

entrez_assemblies_df = pd.DataFrame(all_rows)

if not entrez_assemblies_df.empty:
    valid_mask = entrez_assemblies_df["assembly_accession"].fillna("").astype(str).str.len().gt(0)
    valid_rows = entrez_assemblies_df[valid_mask].drop_duplicates(subset=["assembly_accession", "taxon_id"])
    diagnostic_rows = entrez_assemblies_df[~valid_mask]
    entrez_assemblies_df = pd.concat([valid_rows, diagnostic_rows], ignore_index=True)

write_table(entrez_assemblies_df, DIRS["ncbi"] / "ncbi_assemblies_manifest_entrez_assembly")
print("Entrez Assembly rows:", len(entrez_assemblies_df))
print("Valid assembly rows:", int(entrez_assemblies_df["assembly_accession"].fillna("").astype(str).str.len().gt(0).sum()) if not entrez_assemblies_df.empty else 0)
entrez_assemblies_df.head()

In [ ]:
# ============================================================
# 9C. Select best assembly manifest and generate optional download commands
# ============================================================

if not datasets_assemblies_df.empty:
    assemblies_best_df = datasets_assemblies_df.copy()
    assembly_source_used = "ncbi_datasets_cli"
elif not entrez_assemblies_df.empty:
    assemblies_best_df = entrez_assemblies_df.copy()
    assembly_source_used = "entrez_assembly"
else:
    assemblies_best_df = pd.DataFrame()
    assembly_source_used = "none"

if not assemblies_best_df.empty and "assembly_accession" in assemblies_best_df.columns:
    valid = assemblies_best_df["assembly_accession"].fillna("").astype(str).str.len().gt(0)
    assemblies_best_df = pd.concat([
        assemblies_best_df[valid].drop_duplicates(subset=["assembly_accession", "taxon_id"]),
        assemblies_best_df[~valid]
    ], ignore_index=True)

write_table(assemblies_best_df, DIRS["ncbi"] / "ncbi_assemblies_manifest")
write_table(assemblies_best_df, DIRS["derived"] / "best_available_ncbi_assemblies")

print("Assembly source used:", assembly_source_used)
print("Assembly rows:", len(assemblies_best_df))
print("Valid assembly rows:", int(assemblies_best_df["assembly_accession"].fillna("").astype(str).str.len().gt(0).sum()) if not assemblies_best_df.empty else 0)

def build_datasets_download_command(taxon_name: str, out_zip: pathlib.Path) -> List[str]:
    cmd = [
        "datasets", "download", "genome", "taxon", taxon_name,
        "--include", CONFIG["ncbi_datasets_include"],
        "--assembly-source", CONFIG["ncbi_assembly_source"],
        "--filename", str(out_zip),
        "--no-progressbar",
    ]
    if CONFIG.get("ncbi_assembly_level"):
        cmd.extend(["--assembly-level", CONFIG["ncbi_assembly_level"]])
    if CONFIG.get("ncbi_exclude_atypical", True):
        cmd.append("--exclude-atypical")
    if CONFIG.get("ncbi_api_key"):
        cmd.extend(["--api-key", CONFIG["ncbi_api_key"]])
    return cmd

download_commands = []
for _, tax in taxa_df.iterrows():
    out_zip = DIRS["ncbi"] / "genome_packages" / f"ncbi_datasets_{safe_name(tax['scientific_name'])}_{RUN_ID}.zip"
    cmd = build_datasets_download_command(tax["scientific_name"], out_zip)
    download_commands.append({
        "taxon_id": tax["taxon_id"],
        "scientific_name": tax["scientific_name"],
        "command": " ".join(cmd),
        "zip": str(out_zip),
        "datasets_cli_available": executable_exists("datasets"),
        "note": "Download command requires ncbi-datasets-cli. Assembly metadata already collected with Entrez fallback."
    })

genome_download_cmds_df = pd.DataFrame(download_commands)
write_table(genome_download_cmds_df, DIRS["ncbi"] / "genome_package_download_commands")
assemblies_best_df.head()

## 10. ENA run metadata: transcriptome, WGS, amplicon and targeted sequencing

In [ ]:
# ============================================================
# 10. ENA run metadata
# ============================================================

ENA_PORTAL = "https://www.ebi.ac.uk/ena/portal/api/search"

ENA_FIELDS = [
    "run_accession", "experiment_accession", "sample_accession", "secondary_sample_accession",
    "study_accession", "secondary_study_accession", "submission_accession",
    "tax_id", "scientific_name", "sample_title", "study_title", "experiment_title",
    "library_strategy", "library_source", "library_selection", "library_layout",
    "instrument_platform", "instrument_model",
    "fastq_ftp", "fastq_md5", "fastq_bytes",
    "submitted_ftp", "submitted_md5", "submitted_bytes",
    "base_count", "read_count", "first_public", "last_updated",
    "country", "collection_date", "isolate", "cultivar", "ecotype",
    "tissue_lib", "dev_stage", "sample_description", "center_name", "broker_name"
]

def ena_search_read_run(taxid: int, include_descendants: bool = True, limit: int = 5000) -> pd.DataFrame:
    op = "tax_tree" if include_descendants else "tax_eq"
    query = f"{op}({taxid})"
    params = {
        "result": "read_run",
        "query": query,
        "fields": ",".join(ENA_FIELDS),
        "format": "tsv",
        "limit": limit,
    }
    r = request_get(ENA_PORTAL, params=params, timeout=300)
    if not r.text.strip():
        return pd.DataFrame()
    df = pd.read_csv(io.StringIO(r.text), sep="\t", dtype=str)
    return add_provenance(df, "ENA Portal API read_run", query)

ena_tables = []
for _, tax in taxa_df.dropna(subset=["ncbi_taxid"]).iterrows():
    df = ena_search_read_run(int(tax["ncbi_taxid"]), include_descendants=True, limit=CONFIG["ena_limit_per_taxon"])
    if not df.empty:
        df["taxon_id"] = tax["taxon_id"]
        df["input_taxon"] = tax["scientific_name"]
        df["resolved_taxid"] = int(tax["ncbi_taxid"])
        terms = [x.lower() for x in CONFIG["ena_library_strategy_terms"]]
        df["is_relevant_strategy"] = df["library_strategy"].fillna("").str.lower().apply(lambda x: any(t.lower() in x for t in terms))
        df["is_query_capped"] = int(len(df) >= CONFIG["ena_limit_per_taxon"])
        ena_tables.append(df)

ena_df = pd.concat(ena_tables, ignore_index=True) if ena_tables else pd.DataFrame()
if not ena_df.empty:
    ena_df = ena_df.drop_duplicates(subset=["run_accession"])
write_table(ena_df, DIRS["ena"] / "ena_run_manifest")
ena_df.head()

## 11. UniProt seed proteins for candidate pathway genes

In [ ]:
# ============================================================
# 11. UniProt seed proteins
# ============================================================

UNIPROT_SEARCH = "https://rest.uniprot.org/uniprotkb/search"

UNIPROT_FIELDS = [
    "accession", "id", "protein_name", "gene_names", "organism_name", "organism_id",
    "reviewed", "length", "cc_function", "xref_pfam", "xref_interpro", "xref_kegg"
]

def uniprot_search_tsv(query: str, size: int = 100) -> pd.DataFrame:
    params = {"query": query, "fields": ",".join(UNIPROT_FIELDS), "format": "tsv", "size": size}
    r = request_get(UNIPROT_SEARCH, params=params, timeout=120)
    if not r.text.strip():
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(r.text), sep="\t", dtype=str)

def uniprot_search_fasta(query: str, size: int = 100) -> str:
    params = {"query": query, "format": "fasta", "size": size}
    r = request_get(UNIPROT_SEARCH, params=params, timeout=120)
    return r.text

seed_tables = []
for pwy in PATHWAY_GENE_SETS:
    for term in pwy["query_terms"]:
        q = f'("{term}") AND (taxonomy_id:33090)'
        df = uniprot_search_tsv(q, size=CONFIG["uniprot_size_per_query"])
        if not df.empty:
            df["pathway_id"] = pwy["pathway_id"]
            df["pathway_name"] = pwy["pathway_name"]
            df["gene_query_term"] = term
            df["search_query"] = q
            df = add_provenance(df, "UniProtKB REST API", q)
            seed_tables.append(df)
        time.sleep(0.3)

uniprot_seed_df = pd.concat(seed_tables, ignore_index=True) if seed_tables else pd.DataFrame()
if not uniprot_seed_df.empty:
    entry_col = "Entry" if "Entry" in uniprot_seed_df.columns else "accession"
    if entry_col in uniprot_seed_df.columns:
        uniprot_seed_df = uniprot_seed_df.drop_duplicates(subset=[entry_col, "pathway_id", "gene_query_term"])
write_table(uniprot_seed_df, DIRS["uniprot"] / "uniprot_pathway_seed_proteins")
uniprot_seed_df.head()

In [ ]:
# Optional UniProt seed FASTA download.

if CONFIG["download_uniprot_seed_fastas"]:
    rows = []
    for pwy in PATHWAY_GENE_SETS:
        for term in pwy["query_terms"]:
            q = f'("{term}") AND (taxonomy_id:33090)'
            fasta = uniprot_search_fasta(q, size=CONFIG["uniprot_size_per_query"])
            out = DIRS["uniprot"] / "seed_fastas" / f"{safe_name(pwy['pathway_id'])}__{safe_name(term)}.fasta"
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_text(fasta, encoding="utf-8")
            rows.append({
                "pathway_id": pwy["pathway_id"],
                "gene_query_term": term,
                "query": q,
                "file": str(out),
                "bytes": out.stat().st_size,
                "sha256": sha256_file(out),
                "retrieved_utc": utc_now(),
                "run_id": RUN_ID,
            })
            time.sleep(0.3)
    uniprot_fasta_manifest_df = pd.DataFrame(rows)
    write_table(uniprot_fasta_manifest_df, DIRS["uniprot"] / "uniprot_seed_fasta_manifest")
    display(uniprot_fasta_manifest_df.head())
else:
    print("UniProt seed FASTA download disabled.")

## 12. NCBI Protein: candidate enzyme-family manifest

In [ ]:
# ============================================================
# 12. NCBI Protein candidate enzyme-family manifest
# ============================================================

def protein_query(taxid: int, term: str) -> str:
    # Keep broad [All Fields] fallback because plant proteins are inconsistently annotated.
    return f"txid{taxid}[Organism:exp] AND ({term}[Protein Name] OR {term}[All Fields])"

protein_tables = []
for _, tax in taxa_df.dropna(subset=["ncbi_taxid"]).iterrows():
    for pwy in PATHWAY_GENE_SETS:
        for term in pwy["query_terms"]:
            q = protein_query(int(tax["ncbi_taxid"]), term)
            df = iter_entrez_summaries(
                "protein", q,
                CONFIG["max_entrez_records_per_query"],
                CONFIG["entrez_batch_size"]
            )
            if not df.empty:
                df["taxon_id"] = tax["taxon_id"]
                df["input_taxon"] = tax["scientific_name"]
                df["pathway_id"] = pwy["pathway_id"]
                df["pathway_name"] = pwy["pathway_name"]
                df["gene_query_term"] = term
                df = add_provenance(df, "NCBI Protein via Entrez", q)
                protein_tables.append(df)

ncbi_protein_df = pd.concat(protein_tables, ignore_index=True) if protein_tables else pd.DataFrame()
if not ncbi_protein_df.empty:
    ncbi_protein_df = ncbi_protein_df.drop_duplicates(subset=["accession_version", "pathway_id", "gene_query_term"])
write_table(ncbi_protein_df, DIRS["ncbi"] / "ncbi_candidate_protein_manifest")
ncbi_protein_df.head()

## 13. Evidence-tiered candidate gene table and pathway features

This section fixes the noisy-pathway issue. Generic family hits such as UGT, CYP450, TPS, peroxidase and reductase are automatically downgraded unless the title/query includes more specific terms.

In [ ]:
# ============================================================
# 13A. Evidence-tiered candidate gene table
# ============================================================

def normalize_text(x):
    return str(x).lower() if pd.notna(x) else ""

def assign_rule_based_evidence_tier(row):
    title = normalize_text(row.get("title", ""))
    term = normalize_text(row.get("gene_query_term", ""))

    strong_specific_terms = [
        "4-hydroxyphenylacetaldehyde synthase",
        "4-hydroxyphenylacetaldehyde reductase",
        "tyrosol glucosyltransferase",
        "ugt73b6",
        "amorpha-4,11-diene synthase",
        "taxadiene synthase",
        "strictosidine synthase",
        "norcoclaurine synthase",
        "berberine bridge enzyme",
        "chalcone synthase",
        "phenylalanine ammonia-lyase",
        "beta-amyrin synthase",
        "oxidosqualene cyclase",
    ]

    broad_family_terms = [
        "udp-glycosyltransferase",
        "cytochrome p450",
        "terpene synthase",
        "peroxidase",
        "o-methyltransferase",
        "reductase",
        "dehydrogenase",
        "transferase",
    ]

    if any(s in title or s in term for s in strong_specific_terms):
        return "B_or_C_specific_candidate"

    if any(b in title or b in term for b in broad_family_terms):
        return "D_broad_family_candidate"

    if normalize_text(row.get("pathway_id", "")):
        return "C_candidate_by_query_context"

    return "E_weak_or_unclear"

def specificity_score(row):
    title = normalize_text(row.get("title", ""))
    term = normalize_text(row.get("gene_query_term", ""))

    score = 0
    specific_words = [
        "salidroside", "tyrosol", "hypericin", "artemisinin", "paclitaxel",
        "glycyrrhizin", "chalcone synthase", "strictosidine synthase",
        "taxadiene synthase", "amorpha", "4-hydroxyphenylacetaldehyde",
        "beta-amyrin", "oxidosqualene"
    ]

    broad_or_weak_words = [
        "hypothetical", "predicted", "putative", "like", "uncharacterized",
        "cytochrome p450", "udp-glycosyltransferase", "reductase", "dehydrogenase"
    ]

    for w in specific_words:
        if w in title or w in term:
            score += 2

    for w in broad_or_weak_words:
        if w in title:
            score -= 1

    return score

if ncbi_protein_df.empty:
    candidate_gene_evidence_df = pd.DataFrame()
else:
    keep_cols = [
        "taxon_id", "input_taxon", "resolved_taxid",
        "pathway_id", "pathway_name", "gene_query_term",
        "accession", "accession_version", "title",
        "organism", "taxid", "length_bp", "source", "query",
        "query_total_count", "records_retrieved_for_query", "is_query_capped",
        "retrieved_utc", "run_id"
    ]
    keep_cols = [c for c in keep_cols if c in ncbi_protein_df.columns]

    candidate_gene_evidence_df = ncbi_protein_df[keep_cols].copy()
    candidate_gene_evidence_df["evidence_tier_auto"] = candidate_gene_evidence_df.apply(assign_rule_based_evidence_tier, axis=1)
    candidate_gene_evidence_df["specificity_score_auto"] = candidate_gene_evidence_df.apply(specificity_score, axis=1)
    candidate_gene_evidence_df["manual_review_required"] = True
    candidate_gene_evidence_df["recommended_use"] = np.where(
        candidate_gene_evidence_df["evidence_tier_auto"].eq("B_or_C_specific_candidate"),
        "candidate_pathway_feature",
        np.where(
            candidate_gene_evidence_df["evidence_tier_auto"].eq("D_broad_family_candidate"),
            "weak_family_count_feature",
            "graph_prior_or_exclude_until_review"
        )
    )

    candidate_gene_evidence_df = candidate_gene_evidence_df.sort_values(
        ["taxon_id", "pathway_id", "specificity_score_auto"],
        ascending=[True, True, False]
    )

write_table(candidate_gene_evidence_df, DIRS["derived"] / "candidate_gene_evidence_tiered")
candidate_gene_evidence_df.head(20)

In [ ]:
# ============================================================
# 13B. Summarize pathway evidence features by taxon
# ============================================================

if candidate_gene_evidence_df.empty:
    pathway_feature_df = pd.DataFrame()
else:
    grouped = []
    for (taxon_id, pathway_id), sub in candidate_gene_evidence_df.groupby(["taxon_id", "pathway_id"], dropna=False):
        n_total = len(sub)
        n_specific = int(sub["evidence_tier_auto"].eq("B_or_C_specific_candidate").sum())
        n_broad = int(sub["evidence_tier_auto"].eq("D_broad_family_candidate").sum())
        max_specificity = float(pd.to_numeric(sub["specificity_score_auto"], errors="coerce").max())

        # Conservative pathway potential score.
        pathway_potential_score = (
            min(n_specific, 10) * 2.0 +
            min(n_broad, 20) * 0.25 +
            max(0, max_specificity)
        )

        grouped.append({
            "taxon_id": taxon_id,
            "pathway_id": pathway_id,
            "n_candidate_protein_hits": n_total,
            "n_specific_candidate_hits": n_specific,
            "n_broad_family_hits": n_broad,
            "max_specificity_score": max_specificity,
            "pathway_potential_score": pathway_potential_score,
            "evidence_strength_class": (
                "strong_candidate" if n_specific >= 3 else
                "moderate_candidate" if n_specific >= 1 else
                "weak_family_only" if n_broad > 0 else
                "none"
            ),
            "source": "derived from candidate_gene_evidence_tiered",
            "retrieved_utc": utc_now(),
            "run_id": RUN_ID,
        })

    pathway_feature_df = pd.DataFrame(grouped)

write_table(pathway_feature_df, DIRS["derived"] / "pathway_features_by_taxon")
pathway_feature_df.head()

## 14. GBIF occurrence context and filtering

This section fixes the previous GBIF issue. The output now includes:

- raw occurrence manifest;
- liberal coordinate-filtered occurrence table;
- strict coordinate-filtered occurrence table;
- strict specimen-focused table;
- QC report by taxon and basis of record.

Use `gbif_occurrence_strict_specimen.tsv` for conservative environmental validation.  
Use `gbif_occurrence_liberal_filtered.tsv` for broader environmental-range modelling.

In [ ]:
# ============================================================
# 14A. GBIF occurrence download
# ============================================================

GBIF_OCCURRENCE_SEARCH = "https://api.gbif.org/v1/occurrence/search"
GBIF_SPECIES_MATCH = "https://api.gbif.org/v1/species/match"

def gbif_match_taxon(name: str) -> Dict:
    try:
        r = request_get(GBIF_SPECIES_MATCH, params={"name": name}, timeout=60)
        return r.json()
    except Exception as e:
        return {"error": str(e)}

def gbif_occurrences(scientific_name: str, limit: int = 1000, country: str = "") -> pd.DataFrame:
    match = gbif_match_taxon(scientific_name)
    taxon_key = match.get("usageKey") or match.get("acceptedUsageKey")
    if not taxon_key:
        return pd.DataFrame([{"scientific_name": scientific_name, "gbif_error": match.get("error", "no taxonKey")}])

    all_rows = []
    page_limit = min(300, limit)
    offset = 0

    while offset < limit:
        params = {
            "taxonKey": taxon_key,
            "hasCoordinate": "true",
            "limit": page_limit,
            "offset": offset,
        }
        if country:
            params["country"] = country

        r = request_get(GBIF_OCCURRENCE_SEARCH, params=params, timeout=90)
        data = r.json()

        for rec in data.get("results", []):
            all_rows.append({
                "gbif_key": rec.get("key"),
                "taxon_key": taxon_key,
                "scientific_name_query": scientific_name,
                "scientific_name": rec.get("scientificName"),
                "accepted_scientific_name": rec.get("acceptedScientificName"),
                "country": rec.get("country"),
                "country_code": rec.get("countryCode"),
                "decimal_latitude": rec.get("decimalLatitude"),
                "decimal_longitude": rec.get("decimalLongitude"),
                "coordinate_uncertainty_m": rec.get("coordinateUncertaintyInMeters"),
                "event_date": rec.get("eventDate"),
                "basis_of_record": rec.get("basisOfRecord"),
                "institution_code": rec.get("institutionCode"),
                "dataset_key": rec.get("datasetKey"),
                "has_geospatial_issue": rec.get("hasGeospatialIssues"),
                "last_interpreted": rec.get("lastInterpreted"),
                "gbif_match_taxon_key": taxon_key,
                "gbif_match_status": match.get("matchType"),
                "gbif_match_confidence": match.get("confidence"),
            })

        if data.get("endOfRecords", False):
            break

        offset += page_limit
        time.sleep(0.2)

    df = pd.DataFrame(all_rows)
    if not df.empty:
        df = add_provenance(df, "GBIF occurrence API", f"taxonKey={taxon_key}; country={country or 'global'}")
    return df

gbif_tables = []
for _, tax in taxa_df.iterrows():
    df = gbif_occurrences(
        tax["scientific_name"],
        limit=CONFIG["gbif_occurrence_limit_per_taxon"],
        country=CONFIG.get("gbif_occurrence_country", "")
    )
    if not df.empty:
        df["taxon_id"] = tax["taxon_id"]
        df["input_taxon"] = tax["scientific_name"]
        df["ml_role"] = tax.get("ml_role", "")
        gbif_tables.append(df)

gbif_df = pd.concat(gbif_tables, ignore_index=True) if gbif_tables else pd.DataFrame()
if not gbif_df.empty and "gbif_key" in gbif_df.columns:
    gbif_df = gbif_df.drop_duplicates(subset=["gbif_key"])
write_table(gbif_df, DIRS["gbif"] / "gbif_occurrence_manifest")
gbif_df.head()

In [ ]:
# ============================================================
# 14B. GBIF coordinate QC and filtered outputs
# ============================================================

def gbif_filter_table(gbif_df: pd.DataFrame, uncertainty_threshold_m: int, specimen_only: bool = False) -> pd.DataFrame:
    if gbif_df.empty:
        return pd.DataFrame()

    df = gbif_df.copy()

    df["decimal_latitude"] = pd.to_numeric(df["decimal_latitude"], errors="coerce")
    df["decimal_longitude"] = pd.to_numeric(df["decimal_longitude"], errors="coerce")
    df["coordinate_uncertainty_m"] = pd.to_numeric(df["coordinate_uncertainty_m"], errors="coerce")

    has_coords = df["decimal_latitude"].notna() & df["decimal_longitude"].notna()
    valid_lat = df["decimal_latitude"].between(-90, 90)
    valid_lon = df["decimal_longitude"].between(-180, 180)

    geospatial_issue = df["has_geospatial_issue"].astype(str).str.lower().isin(["true", "1", "yes"])
    uncertainty_ok = df["coordinate_uncertainty_m"].isna() | (df["coordinate_uncertainty_m"] <= uncertainty_threshold_m)

    keep = has_coords & valid_lat & valid_lon & (~geospatial_issue) & uncertainty_ok

    if specimen_only:
        keep = keep & df["basis_of_record"].isin(CONFIG["gbif_preferred_basis_of_record"])

    out = df[keep].copy()
    out["gbif_filter_uncertainty_threshold_m"] = uncertainty_threshold_m
    out["gbif_filter_specimen_only"] = specimen_only
    out["gbif_coordinate_filter_status"] = "passed"
    out["spatial_block_lat"] = np.floor(out["decimal_latitude"] / CONFIG["spatial_block_degrees"]).astype("Int64")
    out["spatial_block_lon"] = np.floor(out["decimal_longitude"] / CONFIG["spatial_block_degrees"]).astype("Int64")
    out["spatial_block_id"] = out["spatial_block_lat"].astype(str) + "_" + out["spatial_block_lon"].astype(str)

    return out

gbif_liberal_df = gbif_filter_table(
    gbif_df,
    uncertainty_threshold_m=CONFIG["gbif_liberal_coordinate_uncertainty_m"],
    specimen_only=False
)

gbif_strict_df = gbif_filter_table(
    gbif_df,
    uncertainty_threshold_m=CONFIG["gbif_strict_coordinate_uncertainty_m"],
    specimen_only=False
)

gbif_strict_specimen_df = gbif_filter_table(
    gbif_df,
    uncertainty_threshold_m=CONFIG["gbif_strict_coordinate_uncertainty_m"],
    specimen_only=True
)

write_table(gbif_liberal_df, DIRS["gbif"] / "gbif_occurrence_liberal_filtered")
write_table(gbif_strict_df, DIRS["gbif"] / "gbif_occurrence_strict_filtered")
write_table(gbif_strict_specimen_df, DIRS["gbif"] / "gbif_occurrence_strict_specimen")

# QC summaries
gbif_qc_rows = []
if not gbif_df.empty:
    total = len(gbif_df)
    gbif_qc_rows.append({"metric": "raw_records", "value": total})
    gbif_qc_rows.append({"metric": "liberal_filtered_records", "value": len(gbif_liberal_df)})
    gbif_qc_rows.append({"metric": "strict_filtered_records", "value": len(gbif_strict_df)})
    gbif_qc_rows.append({"metric": "strict_specimen_records", "value": len(gbif_strict_specimen_df)})
    gbif_qc_rows.append({"metric": "human_observation_records", "value": int((gbif_df["basis_of_record"] == "HUMAN_OBSERVATION").sum()) if "basis_of_record" in gbif_df else 0})
    gbif_qc_rows.append({"metric": "missing_coordinate_uncertainty_records", "value": int(pd.to_numeric(gbif_df["coordinate_uncertainty_m"], errors="coerce").isna().sum()) if "coordinate_uncertainty_m" in gbif_df else 0})
    gbif_qc_rows.append({"metric": "records_uncertainty_gt_1km", "value": int((pd.to_numeric(gbif_df["coordinate_uncertainty_m"], errors="coerce") > 1000).sum()) if "coordinate_uncertainty_m" in gbif_df else 0})
    gbif_qc_rows.append({"metric": "records_uncertainty_gt_10km", "value": int((pd.to_numeric(gbif_df["coordinate_uncertainty_m"], errors="coerce") > 10000).sum()) if "coordinate_uncertainty_m" in gbif_df else 0})

gbif_qc_df = pd.DataFrame(gbif_qc_rows)
write_table(gbif_qc_df, DIRS["gbif"] / "gbif_occurrence_qc_summary")

if not gbif_df.empty and "basis_of_record" in gbif_df:
    gbif_basis_qc_df = (
        gbif_df.groupby(["taxon_id", "input_taxon", "basis_of_record"], dropna=False)
        .size()
        .reset_index(name="n_records")
        .sort_values(["taxon_id", "n_records"], ascending=[True, False])
    )
else:
    gbif_basis_qc_df = pd.DataFrame()
write_table(gbif_basis_qc_df, DIRS["gbif"] / "gbif_basis_of_record_qc")

display(gbif_qc_df)
display(gbif_basis_qc_df.head(20))

## 15. PubMed literature manifest for metabolite accumulation and genetic/pathway evidence

In [ ]:
# ============================================================
# 15. PubMed literature manifest
# ============================================================

def pubmed_search_manifest(term: str, max_records: int = 200) -> pd.DataFrame:
    sr = ncbi_esearch("pubmed", term, retmax=max_records)
    ids = sr.get("idlist", [])
    if not ids:
        return pd.DataFrame()
    rows = []
    for i in range(0, len(ids), 100):
        sm = ncbi_esummary("pubmed", ids[i:i+100])
        for uid in sm.get("uids", []):
            rec = sm.get(uid, {})
            articleids = rec.get("articleids", [])
            doi = "; ".join([x.get("value", "") for x in articleids if x.get("idtype") == "doi"])
            rows.append({
                "pmid": uid,
                "title": rec.get("title"),
                "journal": rec.get("fulljournalname"),
                "pubdate": rec.get("pubdate"),
                "doi": doi,
                "authors": "; ".join([a.get("name", "") for a in rec.get("authors", [])[:10]]),
                "query": term,
                "source": "PubMed via Entrez",
                "retrieved_utc": utc_now(),
                "run_id": RUN_ID,
            })
    return pd.DataFrame(rows)

literature_rows = []
for t in TARGET_TAXA:
    for c in TARGET_COMPOUNDS:
        q = (
            f'("{t["scientific_name"]}"[Title/Abstract]) AND '
            f'("{c["compound_name"]}"[Title/Abstract] OR "{c["compound_class"]}"[Title/Abstract]) AND '
            f'(accumulation OR concentration OR content OR metabolite OR biosynthesis OR pathway OR gene OR environment OR stress)'
        )
        df = pubmed_search_manifest(q, max_records=100)
        if not df.empty:
            df["taxon_id"] = t["taxon_id"]
            df["taxon_name"] = t["scientific_name"]
            df["compound_id"] = c["compound_id"]
            df["compound_name"] = c["compound_name"]
            df["compound_record_type"] = c.get("record_type", "")
            df["compound_class"] = c.get("compound_class", "")
            literature_rows.append(df)

literature_df = pd.concat(literature_rows, ignore_index=True) if literature_rows else pd.DataFrame()
if not literature_df.empty:
    literature_df = literature_df.drop_duplicates(subset=["pmid", "taxon_id", "compound_id"])
write_table(literature_df, DIRS["literature"] / "pubmed_metabolite_genetic_literature_manifest")
literature_df.head()

## 16. Taxon–compound target seed table and quantitative curation template

This table is a **seed**, not a final target label table. It helps choose which compound/taxon pairs deserve manual curation.

In [ ]:
# ============================================================
# 16A. Build target seed table from PubMed manifest
# ============================================================

target_compounds_df = pd.DataFrame(TARGET_COMPOUNDS)
target_taxa_df = pd.DataFrame(TARGET_TAXA)

if literature_df.empty:
    taxon_compound_target_seed_df = pd.DataFrame()
else:
    rows = []
    for _, tax in target_taxa_df.iterrows():
        for _, comp in target_compounds_df.iterrows():
            sub = literature_df[
                (literature_df.get("taxon_id", "").astype(str) == str(tax["taxon_id"])) &
                (literature_df.get("compound_id", "").astype(str) == str(comp["compound_id"]))
            ]

            n_papers = len(sub)
            n_recent = 0

            if n_papers > 0 and "pubdate" in sub.columns:
                years = pd.to_numeric(sub["pubdate"].astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
                n_recent = int((years >= 2015).sum())

            title_text = " ".join(sub.get("title", pd.Series(dtype=str)).fillna("").astype(str)).lower()
            has_accumulation_terms = any(word in title_text for word in ["accumulation", "concentration", "content", "metabolite", "biosynthesis", "pathway"])
            has_exact_compound_in_title = str(comp["compound_name"]).lower() in title_text if n_papers else False

            # Avoid treating every cross-pair as biologically true.
            if n_papers >= 3 and has_accumulation_terms and has_exact_compound_in_title:
                recommendation = "candidate_for_quantitative_curation"
                confidence = "moderate"
            elif n_papers > 0 and has_exact_compound_in_title:
                recommendation = "binary_presence_or_literature_support"
                confidence = "low"
            elif n_papers > 0:
                recommendation = "literature_noise_or_indirect_context"
                confidence = "very_low"
            else:
                recommendation = "no_target_support_yet"
                confidence = "none"

            rows.append({
                "taxon_id": tax["taxon_id"],
                "scientific_name": tax["scientific_name"],
                "family": tax.get("family", ""),
                "rank": tax.get("rank", ""),
                "ml_role": tax.get("ml_role", ""),
                "compound_id": comp["compound_id"],
                "compound_name": comp["compound_name"],
                "compound_class": comp["compound_class"],
                "compound_record_type": comp.get("record_type", ""),
                "n_supporting_pubmed_records": n_papers,
                "n_recent_pubmed_records_2015plus": n_recent,
                "has_literature_support": int(n_papers > 0),
                "has_accumulation_or_pathway_terms": int(has_accumulation_terms),
                "has_exact_compound_in_title": int(has_exact_compound_in_title),
                "target_type_recommendation": recommendation,
                "target_confidence_initial": confidence,
                "requires_manual_curation": True,
                "source": "derived from PubMed literature manifest",
                "retrieved_utc": utc_now(),
                "run_id": RUN_ID,
            })

    taxon_compound_target_seed_df = pd.DataFrame(rows)

write_table(taxon_compound_target_seed_df, DIRS["derived"] / "taxon_compound_target_seed_table")
taxon_compound_target_seed_df.head(20)

In [ ]:
# ============================================================
# 16B. Manual curation template for quantitative metabolite values
# ============================================================

metabolite_curation_columns = [
    "curation_id",
    "taxon_id",
    "scientific_name",
    "compound_id",
    "compound_name",
    "plant_part",
    "developmental_stage",
    "environment_or_treatment",
    "country_or_region",
    "latitude",
    "longitude",
    "value",
    "unit",
    "assay_method",
    "extraction_method",
    "sample_basis",
    "pmid",
    "doi",
    "reference_url",
    "linkage_level",
    "curation_confidence",
    "notes",
    "curator",
    "curated_utc",
]

metabolite_measurement_template_df = pd.DataFrame(columns=metabolite_curation_columns)
write_table(metabolite_measurement_template_df, DIRS["derived"] / "manual_metabolite_measurement_curation_template")
print("Manual metabolite measurement curation template written.")

## 17. LOTUS natural-product evidence placeholder

This notebook writes a schema and manifest. In a later version, use a downloaded LOTUS release or API-specific parser and normalize taxon/compound names.

In [ ]:
# ============================================================
# 17. LOTUS source placeholder
# ============================================================

lotus_expected_schema = pd.DataFrame([
    {"field": "lotus_id", "description": "LOTUS compound identifier"},
    {"field": "compound_name", "description": "reported natural product name"},
    {"field": "inchikey", "description": "structure key for compound harmonization"},
    {"field": "smiles", "description": "structure string"},
    {"field": "organism_name", "description": "reported organism/taxon association"},
    {"field": "organism_taxonomy", "description": "taxonomy string if available"},
    {"field": "reference", "description": "publication or evidence source"},
    {"field": "evidence_quality", "description": "manual or source-derived confidence flag"},
])
write_table(lotus_expected_schema, DIRS["compounds"] / "lotus_expected_schema")

lotus_manifest = pd.DataFrame([{
    "source": "LOTUS natural products database",
    "recommended_action": "download complete release or query API for compound/taxon evidence",
    "output_table": "lotus_compound_taxon_evidence.tsv",
    "join_keys": "compound_name; InChIKey; organism_name; taxon_id",
    "notes": "Use as compound occurrence evidence, not quantitative concentration."
}])
write_table(lotus_manifest, DIRS["compounds"] / "lotus_source_manifest")
lotus_manifest

## 18. Evidence-tier rules

In [ ]:
# ============================================================
# 18. Evidence-tier rules
# ============================================================

EVIDENCE_TIERS = pd.DataFrame([
    {"tier": "A_validated_same_species", "meaning": "experimentally validated gene/enzyme in same species", "recommended_ml_use": "strong pathway feature", "risk": "low"},
    {"tier": "B_validated_close_relative_or_ortholog", "meaning": "ortholog or close homolog of validated enzyme in close relative; reciprocal hit or phylogeny recommended", "recommended_ml_use": "strong but mark inferred", "risk": "moderate"},
    {"tier": "C_domain_and_motif_candidate", "meaning": "correct protein family/domain/motif but no direct validation", "recommended_ml_use": "candidate feature", "risk": "moderate-high"},
    {"tier": "D_family_level_candidate", "meaning": "broad gene family hit only, e.g. UGT/CYP450/TPS without clade support", "recommended_ml_use": "weak feature or count feature", "risk": "high"},
    {"tier": "E_taxon_or_literature_inferred", "meaning": "pathway or compound expected from literature/taxon but no sequence evidence in this dataset", "recommended_ml_use": "prior/graph edge only", "risk": "high"},
    {"tier": "DATA_AVAILABILITY_ONLY", "meaning": "feature records whether public genetic data exists, not biological mechanism", "recommended_ml_use": "coverage/confidence covariate", "risk": "low if not interpreted mechanistically"},
])
write_table(EVIDENCE_TIERS, DIRS["derived"] / "evidence_tier_rules")
EVIDENCE_TIERS

## 19. Improved taxon-level open genetic feature matrix

This section fixes the previous annotation issue. When assemblies come from Entrez Assembly, annotation status is marked as `not_extracted_from_entrez_assembly`; it is not treated as biological absence.

In [ ]:
# ============================================================
# 19. Improved taxon-level open genetic feature matrix
# ============================================================

feature_rows = []

for _, tax in taxa_df.iterrows():
    taxon_id = tax["taxon_id"]

    row = {
        "taxon_id": taxon_id,
        "scientific_name": tax["scientific_name"],
        "family": tax.get("family", ""),
        "rank": tax.get("rank", ""),
        "priority": tax.get("priority", ""),
        "ml_role": tax.get("ml_role", ""),
        "ncbi_taxid": tax.get("ncbi_taxid", ""),
    }

    # Barcode
    sub_barcode = barcode_df[barcode_df.get("taxon_id", "") == taxon_id] if not barcode_df.empty else pd.DataFrame()
    row["n_public_barcode_records"] = len(sub_barcode)
    row["has_public_barcode_data"] = int(len(sub_barcode) > 0)
    for marker in CONFIG["barcode_markers"]:
        row[f"n_marker_{safe_name(marker)}"] = int((sub_barcode.get("marker_query", pd.Series(dtype=str)) == marker).sum()) if not sub_barcode.empty else 0
    row["any_barcode_query_capped"] = int(pd.to_numeric(sub_barcode.get("is_query_capped", pd.Series(dtype=float)), errors="coerce").fillna(0).astype(int).max()) if not sub_barcode.empty else 0

    # Chloroplast/plastid
    sub_cp = chloroplast_df[chloroplast_df.get("taxon_id", "") == taxon_id] if not chloroplast_df.empty else pd.DataFrame()
    row["n_chloroplast_plastid_records"] = len(sub_cp)
    row["has_chloroplast_or_plastid_genome"] = int(len(sub_cp) > 0)
    row["any_chloroplast_query_capped"] = int(pd.to_numeric(sub_cp.get("is_query_capped", pd.Series(dtype=float)), errors="coerce").fillna(0).astype(int).max()) if not sub_cp.empty else 0

    # Assemblies
    sub_asm = assemblies_best_df[assemblies_best_df.get("taxon_id", "") == taxon_id] if not assemblies_best_df.empty else pd.DataFrame()
    valid_asm = sub_asm[sub_asm.get("assembly_accession", pd.Series(dtype=str)).fillna("").astype(str).str.len().gt(0)] if not sub_asm.empty else pd.DataFrame()
    row["n_public_genome_assemblies"] = len(valid_asm)
    row["has_public_genome"] = int(len(valid_asm) > 0)
    row["assembly_metadata_source"] = ";".join(sorted(set(valid_asm.get("assembly_metadata_source", pd.Series(dtype=str)).dropna().astype(str)))) if not valid_asm.empty else ""
    row["assembly_levels_available"] = ";".join(sorted(set(valid_asm.get("assembly_level", pd.Series(dtype=str)).dropna().astype(str)))) if not valid_asm.empty else ""

    annotation_status_values = sorted(set(valid_asm.get("annotation_metadata_status", pd.Series(dtype=str)).dropna().astype(str))) if not valid_asm.empty else []
    row["annotation_metadata_status"] = ";".join(annotation_status_values)
    row["has_public_annotation_extracted"] = int(any(x == "extracted_from_ncbi_datasets" for x in annotation_status_values))
    row["annotation_not_extracted_flag"] = int(any(x == "not_extracted_from_entrez_assembly" for x in annotation_status_values))

    # ENA
    sub_ena = ena_df[ena_df.get("taxon_id", "") == taxon_id] if not ena_df.empty else pd.DataFrame()
    row["n_ena_runs"] = len(sub_ena)
    if not sub_ena.empty:
        ls = sub_ena.get("library_strategy", pd.Series(dtype=str)).fillna("")
        row["n_ena_rnaseq_runs"] = int(ls.str.contains("RNA-Seq", case=False, regex=False).sum())
        row["n_ena_wgs_runs"] = int(ls.str.contains("WGS", case=False, regex=False).sum())
        row["n_ena_amplicon_runs"] = int(ls.str.contains("AMPLICON", case=False, regex=False).sum())
    else:
        row["n_ena_rnaseq_runs"] = 0
        row["n_ena_wgs_runs"] = 0
        row["n_ena_amplicon_runs"] = 0
    row["has_transcriptome_evidence"] = int(row["n_ena_rnaseq_runs"] > 0)

    # Pathway features
    sub_pwy = pathway_feature_df[pathway_feature_df.get("taxon_id", "") == taxon_id] if not pathway_feature_df.empty else pd.DataFrame()
    row["n_pathways_with_candidate_hits"] = int((pd.to_numeric(sub_pwy.get("n_candidate_protein_hits", pd.Series(dtype=float)), errors="coerce").fillna(0) > 0).sum()) if not sub_pwy.empty else 0
    row["n_pathways_with_specific_candidates"] = int((pd.to_numeric(sub_pwy.get("n_specific_candidate_hits", pd.Series(dtype=float)), errors="coerce").fillna(0) > 0).sum()) if not sub_pwy.empty else 0
    row["total_candidate_protein_hits"] = int(pd.to_numeric(sub_pwy.get("n_candidate_protein_hits", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not sub_pwy.empty else 0
    row["total_specific_candidate_hits"] = int(pd.to_numeric(sub_pwy.get("n_specific_candidate_hits", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not sub_pwy.empty else 0
    row["sum_pathway_potential_score"] = float(pd.to_numeric(sub_pwy.get("pathway_potential_score", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not sub_pwy.empty else 0.0
    row["max_pathway_potential_score"] = float(pd.to_numeric(sub_pwy.get("pathway_potential_score", pd.Series(dtype=float)), errors="coerce").fillna(0).max()) if not sub_pwy.empty else 0.0

    # Literature target support
    sub_target = taxon_compound_target_seed_df[taxon_compound_target_seed_df.get("taxon_id", "") == taxon_id] if not taxon_compound_target_seed_df.empty else pd.DataFrame()
    row["n_taxon_compound_pairs_with_literature"] = int(pd.to_numeric(sub_target.get("has_literature_support", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not sub_target.empty else 0
    row["n_taxon_compound_pairs_for_quant_curation"] = int((sub_target.get("target_type_recommendation", pd.Series(dtype=str)) == "candidate_for_quantitative_curation").sum()) if not sub_target.empty else 0
    row["n_taxon_compound_noise_or_indirect"] = int((sub_target.get("target_type_recommendation", pd.Series(dtype=str)) == "literature_noise_or_indirect_context").sum()) if not sub_target.empty else 0

    # Composite scores.
    # Annotation is intentionally not penalized when not extracted.
    row["genetic_data_availability_score"] = (
        row["has_public_barcode_data"] * 1.0 +
        row["has_chloroplast_or_plastid_genome"] * 1.0 +
        row["has_public_genome"] * 2.0 +
        row["has_transcriptome_evidence"] * 2.0
    )

    row["pathway_support_score"] = (
        row["n_pathways_with_specific_candidates"] * 2.0 +
        row["n_pathways_with_candidate_hits"] * 0.5 +
        min(row["sum_pathway_potential_score"], 50) * 0.1
    )

    # Family rows are never ordinary ML rows.
    if row["ml_role"] == "hierarchy_context" or row["rank"] == "family":
        row["open_data_ml_readiness"] = "hierarchy_context"
    elif row["genetic_data_availability_score"] >= 4 and row["pathway_support_score"] >= 2 and row["n_taxon_compound_pairs_with_literature"] >= 1:
        row["open_data_ml_readiness"] = "core"
    elif row["genetic_data_availability_score"] >= 2 or row["n_taxon_compound_pairs_with_literature"] >= 1:
        row["open_data_ml_readiness"] = "support"
    else:
        row["open_data_ml_readiness"] = "background"

    row["source"] = "derived improved open genetic feature matrix"
    row["retrieved_utc"] = utc_now()
    row["run_id"] = RUN_ID

    feature_rows.append(row)

open_genetic_feature_matrix_v2_df = pd.DataFrame(feature_rows)
write_table(open_genetic_feature_matrix_v2_df, DIRS["derived"] / "open_genetic_feature_matrix_by_taxon_v2")
open_genetic_feature_matrix_v2_df[
    ["taxon_id", "scientific_name", "rank", "ml_role", "n_public_genome_assemblies", "has_public_genome", "annotation_metadata_status", "genetic_data_availability_score", "open_data_ml_readiness"]
]

## 20. ML taxon priority table

In [ ]:
# ============================================================
# 20. ML taxon priority table
# ============================================================

if open_genetic_feature_matrix_v2_df.empty:
    ml_taxon_priority_df = pd.DataFrame()
else:
    ml_taxon_priority_df = open_genetic_feature_matrix_v2_df.copy()

    ml_taxon_priority_df["ml_priority_rank_score"] = (
        ml_taxon_priority_df["genetic_data_availability_score"].fillna(0).astype(float) * 2.0 +
        ml_taxon_priority_df["pathway_support_score"].fillna(0).astype(float) * 2.0 +
        ml_taxon_priority_df["n_taxon_compound_pairs_with_literature"].fillna(0).astype(float) * 1.5 +
        ml_taxon_priority_df["n_taxon_compound_pairs_for_quant_curation"].fillna(0).astype(float) * 2.0
    )

    # Family/hierarchy rows should not outrank genus/species candidates.
    ml_taxon_priority_df.loc[
        ml_taxon_priority_df["open_data_ml_readiness"] == "hierarchy_context",
        "ml_priority_rank_score"
    ] = np.nan

    ml_taxon_priority_df["recommended_next_action"] = ml_taxon_priority_df["open_data_ml_readiness"].map({
        "core": "Use in first ML prototype; curate compound targets.",
        "support": "Use as graph/context taxon; improve target or pathway evidence.",
        "background": "Do not use as target row yet; keep for taxonomy/background only.",
        "hierarchy_context": "Use only as graph hierarchy/background; do not use as ordinary ML sample."
    })

    readiness_order = {"core": 0, "support": 1, "background": 2, "hierarchy_context": 3}
    ml_taxon_priority_df["readiness_order"] = ml_taxon_priority_df["open_data_ml_readiness"].map(readiness_order)
    ml_taxon_priority_df = ml_taxon_priority_df.sort_values(
        ["readiness_order", "ml_priority_rank_score"],
        ascending=[True, False]
    ).drop(columns=["readiness_order"])

write_table(ml_taxon_priority_df, DIRS["derived"] / "ml_taxon_priority_table")
ml_taxon_priority_df[
    [
        "taxon_id", "scientific_name", "rank", "open_data_ml_readiness",
        "ml_priority_rank_score", "genetic_data_availability_score",
        "pathway_support_score", "n_taxon_compound_pairs_with_literature",
        "recommended_next_action"
    ]
]

## 21. Graph-node and graph-edge tables for later GNN / Graph Transformer

In [ ]:
# ============================================================
# 21. Enriched graph tables
# ============================================================

nodes = []
edges = []

def add_node(node_id, node_type, label, **attrs):
    rec = {"node_id": node_id, "node_type": node_type, "label": label}
    rec.update(attrs)
    nodes.append(rec)

def add_edge(source_id, target_id, edge_type, weight=1.0, **attrs):
    rec = {"source_id": source_id, "target_id": target_id, "edge_type": edge_type, "weight": weight}
    rec.update(attrs)
    edges.append(rec)

for _, row in taxa_df.iterrows():
    add_node(
        f"taxon:{row['taxon_id']}",
        "taxon",
        row["scientific_name"],
        family=row.get("family", ""),
        rank=row.get("rank", ""),
        ml_role=row.get("ml_role", ""),
        ncbi_taxid=row.get("ncbi_taxid", ""),
    )

for comp in TARGET_COMPOUNDS:
    add_node(
        f"compound:{comp['compound_id']}",
        "compound" if comp.get("record_type") == "molecule" else "compound_class",
        comp["compound_name"],
        compound_class=comp["compound_class"],
        record_type=comp.get("record_type", ""),
    )

for pwy in PATHWAY_GENE_SETS:
    add_node(f"pathway:{pwy['pathway_id']}", "pathway", pwy["pathway_name"])
    for comp_id in str(pwy["compound_ids"]).split(";"):
        comp_id = comp_id.strip()
        if comp_id:
            add_edge(f"compound:{comp_id}", f"pathway:{pwy['pathway_id']}", "compound_belongs_to_pathway")
    for gf in str(pwy["gene_families"]).split(";"):
        gf = gf.strip()
        if gf:
            add_node(f"gene_family:{gf}", "gene_family", gf)
            add_edge(f"pathway:{pwy['pathway_id']}", f"gene_family:{gf}", "pathway_uses_gene_family")

if not pathway_feature_df.empty:
    for _, row in pathway_feature_df.iterrows():
        add_edge(
            f"taxon:{row['taxon_id']}",
            f"pathway:{row['pathway_id']}",
            "taxon_has_candidate_pathway_evidence",
            weight=float(row.get("pathway_potential_score", 0) or 0),
            n_candidate_protein_hits=row.get("n_candidate_protein_hits", ""),
            n_specific_candidate_hits=row.get("n_specific_candidate_hits", ""),
            evidence_strength_class=row.get("evidence_strength_class", ""),
        )

if not taxon_compound_target_seed_df.empty:
    for _, row in taxon_compound_target_seed_df.iterrows():
        if int(row.get("has_literature_support", 0)) == 1:
            add_edge(
                f"taxon:{row['taxon_id']}",
                f"compound:{row['compound_id']}",
                "taxon_has_literature_compound_evidence",
                weight=float(row.get("n_supporting_pubmed_records", 1) or 1),
                target_type_recommendation=row.get("target_type_recommendation", ""),
                target_confidence_initial=row.get("target_confidence_initial", ""),
            )

if not open_genetic_feature_matrix_v2_df.empty:
    for _, row in open_genetic_feature_matrix_v2_df.iterrows():
        tax_node = f"taxon:{row['taxon_id']}"
        availability_features = {
            "public_barcode_data": row.get("n_public_barcode_records", 0),
            "chloroplast_plastid_data": row.get("n_chloroplast_plastid_records", 0),
            "public_genome_data": row.get("n_public_genome_assemblies", 0),
            "public_transcriptome_data": row.get("n_ena_rnaseq_runs", 0),
        }
        for feature_name, weight in availability_features.items():
            add_node(f"availability:{feature_name}", "availability_feature", feature_name)
            if float(weight or 0) > 0:
                add_edge(tax_node, f"availability:{feature_name}", "taxon_has_open_data_type", weight=float(weight))

graph_nodes_df = pd.DataFrame(nodes).drop_duplicates(subset=["node_id"])
graph_edges_df = pd.DataFrame(edges).drop_duplicates()

graph_nodes_df = add_provenance(graph_nodes_df, "derived enriched graph", "taxon-compound-pathway-gene-open-data graph")
graph_edges_df = add_provenance(graph_edges_df, "derived enriched graph", "taxon-compound-pathway-gene-open-data graph")

write_table(graph_nodes_df, DIRS["derived"] / "graph_nodes_enriched_open_genetic_layer")
write_table(graph_edges_df, DIRS["derived"] / "graph_edges_enriched_open_genetic_layer")
display(graph_nodes_df.head())
display(graph_edges_df.head())

## 22. ML open genetic layer specification

In [ ]:
# ============================================================
# 22. ML open genetic layer specification
# ============================================================

ml_layer_spec = pd.DataFrame([
    {
        "feature_block": "genetic_availability_features",
        "source_tables": "barcode_sequence_manifest; chloroplast_plastid_manifest; ncbi_assemblies_manifest; ena_run_manifest",
        "example_features": "has_public_genome; n_barcode_records; has_chloroplast_plastome; n_ena_rnaseq_runs",
        "recommended_use": "coverage/confidence covariates and broad genetic evidence",
        "risk": "low"
    },
    {
        "feature_block": "barcode_features",
        "source_tables": "barcode FASTA files after optional download",
        "example_features": "haplotype count; nucleotide diversity; marker distance; barcode cluster",
        "recommended_use": "taxonomic/phylogeographic genetic signal when sequence alignment is performed",
        "risk": "moderate; public sequences may not match target population"
    },
    {
        "feature_block": "pathway_candidate_features",
        "source_tables": "uniprot_pathway_seed_proteins; ncbi_candidate_protein_manifest; candidate_gene_evidence_tiered",
        "example_features": "candidate family hit count; evidence tier; specificity score; broad-family count",
        "recommended_use": "medicinal-metabolite biosynthetic potential",
        "risk": "moderate/high unless evidence-tiered"
    },
    {
        "feature_block": "transcriptome_features",
        "source_tables": "ena_run_manifest",
        "example_features": "RNA-seq availability; tissue/treatment labels; expression possible flag",
        "recommended_use": "pathway activity context if expression quantification is later performed",
        "risk": "moderate; sample metadata may be heterogeneous"
    },
    {
        "feature_block": "gbif_occurrence_context",
        "source_tables": "gbif_occurrence_liberal_filtered; gbif_occurrence_strict_specimen",
        "example_features": "coordinates; spatial block; basis of record; coordinate uncertainty",
        "recommended_use": "input for environmental extraction notebook",
        "risk": "human-observation records need sensitivity analysis"
    },
    {
        "feature_block": "graph_features",
        "source_tables": "graph_nodes_enriched_open_genetic_layer; graph_edges_enriched_open_genetic_layer",
        "example_features": "taxon-pathway-compound graph edges; node embeddings; relation weights",
        "recommended_use": "GNN/Graph Transformer notebook",
        "risk": "depends on graph construction choices"
    },
])
write_table(ml_layer_spec, DIRS["derived"] / "ml_open_genetic_layer_specification")
ml_layer_spec

## 23. ML handoff manifest and QC report

In [ ]:
# ============================================================
# 23. ML handoff manifest and QC report
# ============================================================

ML_HANDOFF_FILES = [
    {"file_role": "main_open_genetic_feature_matrix", "path": str(DIRS["derived"] / "open_genetic_feature_matrix_by_taxon_v2.tsv"), "required_for_ml": True, "description": "Primary taxon-level open genetic/pathway feature table."},
    {"file_role": "taxon_priority_table", "path": str(DIRS["derived"] / "ml_taxon_priority_table.tsv"), "required_for_ml": True, "description": "Recommended core/support/background taxa for ML."},
    {"file_role": "candidate_gene_evidence", "path": str(DIRS["derived"] / "candidate_gene_evidence_tiered.tsv"), "required_for_ml": True, "description": "Evidence-tiered candidate pathway protein/gene table."},
    {"file_role": "pathway_features", "path": str(DIRS["derived"] / "pathway_features_by_taxon.tsv"), "required_for_ml": True, "description": "Pathway-level features summarized by taxon."},
    {"file_role": "target_seed_table", "path": str(DIRS["derived"] / "taxon_compound_target_seed_table.tsv"), "required_for_ml": True, "description": "Initial taxon-compound target support from literature."},
    {"file_role": "manual_metabolite_curation_template", "path": str(DIRS["derived"] / "manual_metabolite_measurement_curation_template.tsv"), "required_for_ml": False, "description": "Template for quantitative concentration curation."},
    {"file_role": "enriched_graph_nodes", "path": str(DIRS["derived"] / "graph_nodes_enriched_open_genetic_layer.tsv"), "required_for_ml": False, "description": "Graph nodes for future GNN/Graph Transformer notebook."},
    {"file_role": "enriched_graph_edges", "path": str(DIRS["derived"] / "graph_edges_enriched_open_genetic_layer.tsv"), "required_for_ml": False, "description": "Graph edges for future GNN/Graph Transformer notebook."},
    {"file_role": "gbif_occurrences_raw", "path": str(DIRS["gbif"] / "gbif_occurrence_manifest.tsv"), "required_for_ml": True, "description": "Raw occurrence coordinates."},
    {"file_role": "gbif_occurrences_liberal", "path": str(DIRS["gbif"] / "gbif_occurrence_liberal_filtered.tsv"), "required_for_ml": True, "description": "Liberal coordinate-filtered GBIF records."},
    {"file_role": "gbif_occurrences_strict_specimen", "path": str(DIRS["gbif"] / "gbif_occurrence_strict_specimen.tsv"), "required_for_ml": True, "description": "Strict specimen-focused GBIF records."},
    {"file_role": "compound_dictionary", "path": str(DIRS["compounds"] / "compound_dictionary.tsv"), "required_for_ml": True, "description": "Compound identifiers and compound-class ontology."},
    {"file_role": "assemblies_manifest", "path": str(DIRS["ncbi"] / "ncbi_assemblies_manifest.tsv"), "required_for_ml": True, "description": "Best available public genome assembly metadata."},
]

ml_handoff_manifest_df = pd.DataFrame(ML_HANDOFF_FILES)
ml_handoff_manifest_df["exists_and_nonempty"] = ml_handoff_manifest_df["path"].apply(lambda x: pathlib.Path(x).exists() and pathlib.Path(x).stat().st_size > 0)
ml_handoff_manifest_df["created_utc"] = utc_now()
ml_handoff_manifest_df["run_id"] = RUN_ID
write_table(ml_handoff_manifest_df, DIRS["derived"] / "ml_notebook_handoff_manifest")

qc_lines = []
qc_lines.append("# Notebook 1 QC report")
qc_lines.append(f"Run ID: {RUN_ID}")
qc_lines.append("")
qc_lines.append("## Key outputs")
qc_lines.append(f"- Taxa resolved: {len(taxa_df)}")
qc_lines.append(f"- Barcode records: {len(barcode_df)}")
qc_lines.append(f"- Chloroplast/plastid records: {len(chloroplast_df)}")
qc_lines.append(f"- Assembly rows: {len(assemblies_best_df)}")
qc_lines.append(f"- ENA records: {len(ena_df)}")
qc_lines.append(f"- Candidate protein records: {len(ncbi_protein_df)}")
qc_lines.append(f"- Candidate gene evidence rows: {len(candidate_gene_evidence_df)}")
qc_lines.append(f"- GBIF raw records: {len(gbif_df)}")
qc_lines.append(f"- GBIF liberal records: {len(gbif_liberal_df)}")
qc_lines.append(f"- GBIF strict specimen records: {len(gbif_strict_specimen_df)}")
qc_lines.append(f"- PubMed literature records: {len(literature_df)}")
qc_lines.append("")
qc_lines.append("## Notes")
qc_lines.append("- Family-level rows are marked as hierarchy_context and should not be ordinary ML samples.")
qc_lines.append("- Broad compound classes are not queried against PubChem as single molecules.")
qc_lines.append("- Entrez Assembly fallback marks annotation metadata as not extracted, not absent.")
qc_lines.append("- Generic pathway gene families are evidence-tiered and must be manually reviewed before mechanistic claims.")

qc_path = DIRS["derived"] / "notebook1_qc_report.md"
qc_path.write_text("\n".join(qc_lines), encoding="utf-8")

display(ml_handoff_manifest_df)
print(qc_path)

## 24. Run file manifest

In [ ]:
# ============================================================
# 24. Run file manifest
# ============================================================

manifest_rows = []
for path in PROJECT.rglob("*"):
    if path.is_file():
        try:
            manifest_rows.append({
                "path": str(path.relative_to(PROJECT)),
                "absolute_path": str(path),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
                "modified_utc": dt.datetime.fromtimestamp(path.stat().st_mtime, tz=dt.timezone.utc).isoformat(timespec="seconds"),
                "run_id": RUN_ID,
            })
        except Exception as e:
            print("Skipping manifest entry", path, e)

run_manifest_df = pd.DataFrame(manifest_rows).sort_values("path")
write_table(run_manifest_df, PROJECT / "run_file_manifest")
run_manifest_df.head(20)

## 25. Outputs to use in Notebook 2

Notebook 2 should build the environmental + target ML table.

Primary inputs:

- `derived/open_genetic_feature_matrix_by_taxon_v2.tsv`
- `derived/ml_taxon_priority_table.tsv`
- `derived/pathway_features_by_taxon.tsv`
- `derived/candidate_gene_evidence_tiered.tsv`
- `derived/taxon_compound_target_seed_table.tsv`
- `gbif/gbif_occurrence_liberal_filtered.tsv`
- `gbif/gbif_occurrence_strict_specimen.tsv`
- `compounds/compound_dictionary.tsv`
- `ncbi/ncbi_assemblies_manifest.tsv`

Notebook 2 should create:

- `taxon_occurrence_compound_ml_table.tsv`
- environmental predictor table from WorldClim/CHELSA/SoilGrids or similar
- spatial/taxonomic/compound validation groups

Notebook 3 should train ML models.